# Begin

Courtesy: https://docs.cleanrl.dev/rl-algorithms/ppo-trxl/

In [2]:
# @launchit.collected
# @launchit.collected_temp_config
# @launchit.collected_optuna
# @launchit.collected_initrd
# @launchit.collected_build_docker_launch

In [3]:
# @launchit.collected_manual_run_docker_launch
# @launchit.collected_optuna_run_docker_launch

In [4]:
import os # @launchit.collect
import sys # @launchit.collect
import socket
import copy
from collections import namedtuple, defaultdict, Counter, deque # @launchit.collect
import random
import math
import datetime
import json # @launchit.collect
import pprint # @launchit.collect
import re 
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import pickle # @launchit.collect
import IPython 
from enum import StrEnum, auto # @launchit.collect
import multiprocessing as mp
import queue

import lark # @launchit.collect

from tqdm.notebook import tqdm

import numpy as np # @launchit.collect
import cupy as cp
import einops
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as VF
import torch.optim
import torch.multiprocessing as torch_mp
import torch._dynamo as dynamo
from torch.utils.data import Dataset, DataLoader
from torch.distributions import Categorical

import gymnasium as gym
import ale_py
from moviepy.video.io.ImageSequenceClip import ImageSequenceClip
import av

import optuna # @launchit.collect
from optuna.storages import JournalStorage # @launchit.collect
from optuna.storages.journal import JournalFileBackend # @launchit.collect
from optuna.trial import TrialState

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
build_project_root_path = '${BUILD_PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect
sys.path.append(os.path.join(build_project_root_path, 'lib')) # @launchit.collect
from cleanrl.cleanrl_utils.atari_wrappers import (  # isort:skip
    ClipRewardEnv,
    EpisodicLifeEnv,
    FireResetEnv,
    MaxAndSkipEnv,
    NoopResetEnv,
)
import lang_utils as lu # @launchit.collect
import array_utils as au # @launchit.collect
from math_utils import RecursiveAverageFilter, RecursiveMovingAverageFilter
from logging_utils import *
from artifact_registry import * # @launchit.collect
from torch_utils import *
import launchit 
from hp_utils import * # @launchit.collect
from metrics_collector import RmqSummaryWriter, S3SummaryWriter
from autoincrement import Autoincrement

/home/misha/anaconda3/envs/mine/lib/python3.12/site-packages/cupyx/scipy/__init__.py:10: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  import scipy as _scipy


# Init

In [5]:
# @launchit.collect
class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    DOCKER_LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()

In [6]:
# @launchit.collect_temp_config
# @launchit.disable
# Construct temporary CONFIG object for build and bootstrap purposes
if '${LAUNCHIT_FNAME}' != '$' + '{LAUNCHIT_FNAME}':
    notebook_fname = '${LAUNCHIT_FNAME}'
    notebook_basename = os.path.basename(notebook_fname)
    notebook_name, notebook_ext = os.path.splitext(notebook_basename)
    subproject_name = os.path.basename(os.path.dirname(notebook_fname))
    
    if os.path.exists(os.path.join(project_root_path, '.docker_launch')):
        CONFIG = namedtuple('Config', 'initrd_path, exec_mode')(
            initrd_path=os.path.join(project_root_path, 'run', subproject_name, 'initrd-' + notebook_name),
            exec_mode=ExecMode.DOCKER_LAUNCH_NOTEBOOK,
        )
        assert Logging._instance is None, 'Must create Logging instance with use_raw_stdout, but Logging instance is already created'
        # use_raw_stdout is necessary because sys.stdout is hijacked by papermill and this will lead to problems
        # with string duplications written to stdout by child process (worker)
        # see https://share.google/aimode/ffIcoqyaJdgMsEcoW
        Logging.get(use_raw_stdout=True)    
    else:
        CONFIG = namedtuple('Config', 
                            'initrd_path, relative_initrd_path, relative_run_path, relative_metrics_suite_fname, ' + 
                            'model_group_uri, self_fname, relative_self_fname, self_name, subproject_name, ' + 
                            'docker_registry, exec_mode')(
            initrd_path=os.path.join(build_project_root_path, 'run', subproject_name, 'initrd-' + notebook_name),
            relative_initrd_path=os.path.join('run', subproject_name, 'initrd-' + notebook_name), # relative to build project root
            relative_run_path=os.path.join('run', subproject_name),
            relative_metrics_suite_fname=os.path.join('run', subproject_name, notebook_name + '.metrics_suite.json'),
            model_group_uri='${MODEL_GROUP_URI}',
            self_fname=notebook_fname,
            relative_self_fname=lu.when('/run/' in notebook_fname, 'run/', '') + os.path.join(subproject_name, notebook_basename),  # relative to build project root
            self_name=notebook_name,
            subproject_name=subproject_name,
            docker_registry='cr.selcloud.ru/neurolab',
            exec_mode=ExecMode.LAUNCH_NOTEBOOK,
        )
        os.makedirs(CONFIG.initrd_path, exist_ok=True)

    Logging.get()(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n')
# @launchit.stop

In [7]:
def create_config():
    config = namedtuple('Config', 
                        'host_name, ' +
                        'project_root_path, project_root_uri, model_group_uri, subproject_path, data_path, private_data_path, run_path, initrd_path, ' + 
                        'self_fname, self_name, metrics_suite_fname, ' +
                        'subproject_name,' +
                        'is_cuda, cuda_device, docker_registry, exec_mode, is_interactive')(
        host_name=socket.gethostname(),
        project_root_path=project_root_path,
        project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
        model_group_uri=None,
        subproject_path=os.path.abspath('.'),
        data_path=os.path.join(project_root_path, 'data'),
        private_data_path=None,
        run_path=None,
        initrd_path=None,
        self_fname=None,
        self_name=None,
        metrics_suite_fname=None,
        subproject_name=None,
        is_cuda=torch.cuda.is_available(),
        cuda_device='cuda' if torch.cuda.is_available() else 'cpu',
        docker_registry='cr.selcloud.ru/neurolab',
        exec_mode=ExecMode.MASTER_NOTEBOOK,
        is_interactive=True,
    )
    
    if IPython.get_ipython() is None:
        module_fname = __file__
        module_basename = os.path.basename(module_fname)
        module_name, _ = os.path.splitext(module_basename)
        
        config = config._replace(self_fname=module_fname, self_name=module_name)
        config = config._replace(exec_mode=ExecMode.LAUNCH_MODULE)
    else:
        with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
            notebook_fname = json.load(cf).get('jupyter_session')

            if notebook_fname is None:
                notebook_fname = os.path.join(config.subproject_path, os.path.basename('${LAUNCHIT_FNAME}'))
                assert os.path.exists(notebook_fname)
            
            notebook_basename = os.path.basename(notebook_fname)
            notebook_name, notebook_ext = os.path.splitext(notebook_basename)
        
            m = re.match(r'(\w+)-Copy\d+$', notebook_name)
        
            if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook
    
            config = config._replace(self_fname=notebook_fname, self_name=notebook_name)
            
            is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None

            if is_launch:
                if os.path.exists(os.path.join(project_root_path, '.docker_launch')):
                    config = config._replace(exec_mode=ExecMode.DOCKER_LAUNCH_NOTEBOOK)
                else:
                    config = config._replace(exec_mode=ExecMode.LAUNCH_NOTEBOOK)
            else:
                assert config.exec_mode == ExecMode.MASTER_NOTEBOOK
    
    config = config._replace(is_interactive=config.exec_mode in [ExecMode.MASTER_NOTEBOOK, ExecMode.LAUNCH_NOTEBOOK])
    config = config._replace(subproject_name=os.path.basename(os.path.dirname(config.self_fname)))
    config = config._replace(model_group_uri=f'{config.project_root_uri}.{config.subproject_name}')
    config = config._replace(run_path=os.path.join(project_root_path, 'run', config.subproject_name))
    config = config._replace(initrd_path=os.path.join(project_root_path, 'run', config.subproject_name, 'initrd-' + config.self_name))
    config = config._replace(private_data_path=os.path.join(config.data_path, config.subproject_name))
    config = config._replace(metrics_suite_fname=os.path.join(config.run_path, config.self_name + '.metrics_suite.json'))
    return config

In [8]:
# @launchit.disable_worker
au.init()
LOG = Logging.get()
RNG = np.random.default_rng()
CONFIG = create_config()
LOG.app_name = CONFIG.self_name
LOG.enable('syslog', CONFIG.exec_mode == ExecMode.LAUNCH_MODULE)
LOG.enable('stdout', CONFIG.exec_mode in [ExecMode.MASTER_NOTEBOOK, ExecMode.LAUNCH_NOTEBOOK])
LOG.enable('verbose_stdout', CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK)
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)
os.makedirs(CONFIG.private_data_path, exist_ok=True)
os.makedirs(CONFIG.run_path, exist_ok=True)
os.makedirs(CONFIG.initrd_path, exist_ok=True)

CONFIG=
{'host_name': 'thinkbook',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.17_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/17_rl',
 'data_path': '/home/misha/dev/mine/neurolab/data',
 'private_data_path': '/home/misha/dev/mine/neurolab/data/17_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/17_rl',
 'initrd_path': '/home/misha/dev/mine/neurolab/run/17_rl/initrd-17e_ppo_tr_atari_mp_13',
 'self_fname': '/home/misha/dev/mine/neurolab/17_rl/17e_ppo_tr_atari_mp_13.ipynb',
 'self_name': '17e_ppo_tr_atari_mp_13',
 'metrics_suite_fname': '/home/misha/dev/mine/neurolab/run/17_rl/17e_ppo_tr_atari_mp_13.metrics_suite.json',
 'subproject_name': '17_rl',
 'is_cuda': False,
 'cuda_device': 'cpu',
 'docker_registry': 'cr.selcloud.ru/neurolab',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



# Hyperparameters

In [9]:
# @launchit.disable
# @launchit.collect
class LaunchGoal(StrEnum):
    UNSPECIFIED = auto()
    TRAIN = auto()
    WORKER = auto()

LaunchComponent = namedtuple('LaunchComponent', 'name version uri main_asset_fname')
    
@dataclass(slots=True)
class Hyperparameters:
    # Launch
    launch_goal: LaunchGoal = lu.from_str(LaunchGoal, '${LAUNCH_GOAL}', LaunchGoal.UNSPECIFIED)
    launch_id: int = lu.from_str(int, '${MODEL_VERSION}', 0)

    @dataclass(slots=True)
    class System:
        comment: str = None
        random_seed: int = None
        is_torch_deterministic: bool = True
        is_torch_compile: bool = False

    @dataclass(slots=True)
    class Env:
        ident: str = None
        count: int = 1 # number of parallel game environments
        is_episodic_life: bool = None
        actions_count: int = None # None - consider all actions, otherwise take only first :actions_count
        idle_penalty: float = None
        life_lost_penalty: float = None

    @dataclass(slots=True)
    class Agent:
        parent: str = None # use parent model to initialize weights (continue learning)
        layers_count: int = 3 # number of transformer layers
        heads_count: int = 4 # number of heads used in multi-head attention
        d_model: int = 256 # dimension of the transformer
        obs_sequence_length: int = 10 # observations chain max length
        action_plan_length: int = 10 # number of actions agent must think upfront about
        positional_encoding: str = 'learned' # absolute (sinusoidal), learned

    @dataclass(slots=True)
    class Video:
        capture_policy: str = 'every(1000000)' # video capture policy depending on steps
        capture_preprocessed_obs: bool = False # True - capture preprocessed obs (scaled and turned to grayscale), False - capture obs as is
        capture_env_rams: list = None # list of RAM to record video about
        break_on_level_passed: bool = None # whether to stop recording when level is passed
        
    @dataclass(slots=True)
    class PPO:
        global_steps_count: int = 1_000_000 # total number of steps 
        rollout_steps_count: int = 512 # how many steps to run in a single policy rolllout
        rollout_env_rams: list = None # list of RAM to randomly choose on reset during rollout
        rollout_env_ram_patches: list = None # patches to randomly choose and apply to RAM to get more diverse exploration space
        rollout_game_modifs: list = None
        
        tau: str = None # temperature to inject randomness during actions selection (Gumbel Max)
        
        epochs_count: int = 3 
        minibatches_count: int = 8
        learn_rate: str = None
        optimizer: str = 'AdamW'
        
        vf_coef: float = 0.5 # coefficient of the value function within loss function
        ent_coef: str = None # coefficient of the entropy member  within loss function
        consistency_coef: float = 0.0 # coefficient of the loss for action plan consistency between adjacent steps
        prediction_coef: float = 0.0 # coefficient of the loss for observation emb. prediction, if set to 0.0 the prediction loss is not used
        
        gamma: float = 0.995 # return discount factor gamma
        gae_lambda: float = 0.95 # lambda for the general advantage estimation
        clip_coef: float = 0.1 # the surrogate clipping coefficient
        clip_vloss: bool = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
        max_grad_norm: float = 0.25 # the maximum norm for the gradient clipping
        target_kl: float = None # e target KL divergence threshold
        norm_adv: bool = False # Toggles advantages normalization

    system: System = dataclasses.field(default_factory=System)
    env: Env = dataclasses.field(default_factory=Env)
    agent: Agent = dataclasses.field(default_factory=Agent)
    video: Video = dataclasses.field(default_factory=Video)
    ppo: PPO = dataclasses.field(default_factory=PPO)

    @staticmethod
    def from_dict(d):
        hp = Hyperparameters(**d)
        hp.system = Hyperparameters.System(**hp.system)
        hp.env = Hyperparameters.Env(**hp.env)
        hp.agent = Hyperparameters.Agent(**hp.agent)
        hp.video = Hyperparameters.Video(**hp.video)
        hp.ppo = Hyperparameters.PPO(**hp.ppo)
        return hp

    def _asdict(self):
        return dataclasses.asdict(self)

    def launch_component(self):
        name = lu.when('${MODEL_NAME}' == '$' + '{MODEL_NAME}', CONFIG.self_name, '${MODEL_NAME}')
        return LaunchComponent(name=name, version=self.launch_id,  uri=f'{CONFIG.model_group_uri}.{name}', main_asset_fname=CONFIG.self_fname)

HP = Hyperparameters()

# Launch

## LaunchState

In [10]:
@dataclass(slots=True)
class LaunchState:
    mp_ctx: object = None
    optuna_trial: dict = None
    artifact_registry: object = None
    summary_writer: object = None
    env_observation_space_shape: object = None
    env_action_space_shape: object = None
    agent: object = None
    capture_video_manager: object = None

## Configure

In [11]:
# @launchit.disable
# @launchit.collect
HP.system.random_seed = 42
HP.system.is_torch_deterministic = True
HP.system.is_torch_compile = True
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'system': {'comment': None,
            'random_seed': 42,
            'is_torch_deterministic': True,
            'is_torch_compile': True},
 'env': {'ident': None,
         'count': 1,
         'is_episodic_life': None,
         'actions_count': None,
         'idle_penalty': None,
         'life_lost_penalty': None},
 'agent': {'parent': None,
           'layers_count': 3,
           'heads_count': 4,
           'd_model': 256,
           'obs_sequence_length': 10,
           'action_plan_length': 10,
           'positional_encoding': 'learned'},
 'video': {'capture_policy': 'every(1000000)',
           'capture_preprocessed_obs': False,
           'capture_env_rams': None,
           'break_on_level_passed': None},
 'ppo': {'global_steps_count': 1000000,
         'rollout_steps_count': 512,
         'rollout_env_rams': None,
         'rollout_env_ram_patches': None,
         'rollout_game_modifs': None,
    

## Create

In [12]:
# @launchit.disable_worker
LS = LaunchState()
LS.mp_ctx = torch_mp.get_context('spawn') # Spawn is needed for CUDA, fork doesn't work within PyTorch

LOG(f'HP={HP._asdict()}', when=not CONFIG.is_interactive)
    
if HP.system.random_seed is not None:
    random.seed(HP.system.random_seed)
    torch.manual_seed(HP.system.random_seed)
    RNG = np.random.default_rng(HP.system.random_seed)    
    LOG(f'Random seed={HP.system.random_seed}')

if HP.system.is_torch_deterministic is not None:
    torch.backends.cudnn.deterministic = HP.system.is_torch_deterministic
    LOG(f'{torch.backends.cudnn.deterministic=}')

lc = HP.launch_component()

artifact_registry_type = os.environ.get('ARTIFACT_REGISTRY', '').upper()

if not artifact_registry_type:
    artifact_registry_type = lu.when(CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK, 'S3', 'NEXUS')

match artifact_registry_type:
    case 'NEXUS':
        kwargs = {}

        if os.environ.get('NEXUS_URL', None) is not None:
            kwargs['nexus_url'] = os.environ['NEXUS_URL']

        if os.environ.get('DOWNLOAD_NEXUS_URL', None) is not None:
            kwargs['download_nexus_url'] = os.environ['DOWNLOAD_NEXUS_URL']
        
        LS.artifact_registry = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri, **kwargs)
        LOG(f'Created ArtifactRegistry')
    case 'S3':
        LS.artifact_registry = S3ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)
        LOG(f'Created S3ArtifactRegistry')
    case _:
        assert False, f'Unsupported {artifact_registry_type=}'

if lc.version != 0:
    assert CONFIG.exec_mode != ExecMode.MASTER_NOTEBOOK, 'With MASTER_NOTEBOOK exec_mode one should not overwrite any of the launches (experiments)'
    LS.artifact_registry.attach_asset(lc.name, lc.version, lc.main_asset_fname, replace=True)
    meta = dict(
        hypers=HP._asdict(), 
        config=CONFIG._asdict(), 
    )
    
    with io.StringIO() as b:
        json.dump(meta, b)
        LS.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='meta', replace=True)
else:
    assert CONFIG.exec_mode == ExecMode.MASTER_NOTEBOOK, 'MASTER_NOTEBOOK exec_mode is for working on 0 (dummy) version only'

LS.optuna_trial = None
optuna_trial_fname = os.path.join(CONFIG.initrd_path, 'optuna_trial.json')

if os.path.exists(optuna_trial_fname):
    with open(os.path.join(optuna_trial_fname), 'rt') as f:
        LS.optuna_trial = json.load(f)
        assert 'trial_number' in LS.optuna_trial, LS.optuna_trial
        assert 'study_serial' in LS.optuna_trial, LS.optuna_trial
        assert 'study_name' in LS.optuna_trial, LS.optuna_trial

    LOG(f'Optuna trial loaded from "{optuna_trial_fname}": {LS.optuna_trial}')

summary_log_dir = lc.name

if LS.optuna_trial is not None:
    summary_log_dir = os.path.join(summary_log_dir, f'opt_{LS.optuna_trial['study_serial']}')
    
summary_log_dir = os.path.join(summary_log_dir, str(lc.version))
LOG(f'Tensorboard run={summary_log_dir}')

summary_writer_type = os.environ.get('SUMMARY_WRITER', '').upper()

if not summary_writer_type:
    summary_writer_type = lu.when(CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK, 'S3', 'RMQ')

match summary_writer_type:
    case 'RMQ':
        kwargs = {}

        if os.environ.get('RMQ_CONNECTION_URL', None) is not None:
            kwargs['rmq_connection_url'] = os.environ['RMQ_CONNECTION_URL']

        LS.summary_writer = RmqSummaryWriter(log_dir=summary_log_dir, **kwargs)
        LOG(f'Created RmqSummaryWriter')
    case 'S3':
        LS.summary_writer = S3SummaryWriter(log_dir=summary_log_dir)
        LOG(f'Created S3SummaryWriter')
    case _:
        assert False, f'Unsupported {summary_writer_type=}'

LS.summary_writer.add_text('hyperparameters', pprint.pformat(HP._asdict(), sort_dicts=False), 0)
LS.summary_writer.add_text('config', pprint.pformat(CONFIG._asdict(), sort_dicts=False), 0)
LS.summary_writer.flush()

Random seed=42
torch.backends.cudnn.deterministic=True
Created ArtifactRegistry
Tensorboard run=17e_ppo_tr_atari_mp_13/0
Created RmqSummaryWriter


# Environment

## MyUberWrapper

In [12]:
class MyUberWrapper(gym.vector.VectorWrapper):
    def __init__(self, env, idle_penalty, life_lost_penalty, frame_skip, track_levels):
        super().__init__(env)
        assert isinstance(env, ale_py.AtariVectorEnv)
        self.frame_numbers = np.zeros(env.unwrapped.num_envs, dtype=np.int32)
        self.episode_lengths = np.zeros(env.unwrapped.num_envs, dtype=np.int32)
        self.episode_returns = np.zeros(env.unwrapped.num_envs)
        self.lives = np.zeros(env.unwrapped.num_envs, dtype=np.int32)
        self.was_life_lost = np.zeros(env.unwrapped.num_envs, dtype=np.bool)
        self.life_lengths = np.zeros(env.unwrapped.num_envs, dtype=np.int32)
        self.life_returns = np.zeros(env.unwrapped.num_envs)
        assert idle_penalty <= 0
        assert life_lost_penalty <= 0
        self.idle_penalty = idle_penalty
        self.life_lost_penalty = life_lost_penalty
        self.frame_skip = frame_skip # forward data, used within video capture to compute fps
        self.track_levels = track_levels

        if self.track_levels:
            self.levels = np.zeros(env.unwrapped.num_envs, dtype=np.int32)
            self.temperatures = np.zeros(env.unwrapped.num_envs, dtype=np.uint8) 

    def step(self, actions):
        self.life_lengths[self.was_life_lost] = 0
        self.life_returns[self.was_life_lost] = 0
        
        obs, rewards, terminations, truncations, infos = self.env.step(actions)

        frame_numbers = infos['frame_number'] # grows indefinitely starting from ROM load
        episode_lengths = infos['episode_frame_number'] # resets on every new episode (game)
        lives = infos['lives']

        addon_lenghts = frame_numbers - self.frame_numbers
        self.frame_numbers[:] = frame_numbers
        assert np.all(addon_lenghts >= 0), addon_lenghts
        
        infos['is_episode_over'] = np.logical_or(terminations, truncations)
        self.episode_lengths[:] = episode_lengths
        self.episode_returns += rewards

        infos['is_life_lost'] = np.logical_or(lives < self.lives, infos['is_episode_over'])
        self.was_life_lost[:] = infos['is_life_lost']
        self.lives[:] = lives
        
        self.life_lengths += addon_lenghts
        self.life_returns += rewards

        infos['episode_lengths'] = self.episode_lengths.copy()
        infos['episode_returns'] = self.episode_returns.copy()
        infos['life_lengths'] = self.life_lengths.copy()
        infos['life_returns'] = self.life_returns.copy()

        clipped_rewards = np.sign(rewards)

        if self.idle_penalty < 0:
            clipped_rewards = np.where(clipped_rewards == 0, self.idle_penalty, clipped_rewards)

        if self.life_lost_penalty < 0:
            clipped_rewards[infos['is_life_lost']] = self.life_lost_penalty

        if self.track_levels:
            new_temperatures = infos['rams'][:,101].ravel()
            where_raised = new_temperatures > self.temperatures
            where_not_life_lost = ~infos['is_life_lost']
            where_level_passed = where_raised & where_not_life_lost
            self.levels[where_level_passed] += 1
            self.temperatures = new_temperatures
            infos['level_passed'] = where_level_passed

        return self._fix_obs(obs), clipped_rewards, terminations, truncations, infos

    def reset(self, seed=None, options=None):
        obs, infos = self.env.reset(seed=seed, options=options)
        reset_mask = lu.coalesce(options, {}).get('reset_mask')

        if reset_mask is None:
            reset_mask = np.full(len(self.episode_lengths), True, dtype=np.bool)
            
        self.episode_lengths[reset_mask] = 0
        self.episode_returns[reset_mask] = 0
        self.life_lengths[reset_mask] = 0
        self.life_returns[reset_mask] = 0
        self.was_life_lost[reset_mask] = False

        if self.track_levels:
            self.levels[reset_mask] = 0
            self.temperatures[reset_mask] = infos['rams'][reset_mask,101].ravel()
            
        return self._fix_obs(obs), infos

    def _fix_obs(self, obs):
        # Get rid of dummy dimension cast by degenerate stack frames=1
        assert obs.ndim == 5, obs.shape # [num_envs, stack_size, height, width, 3]
        return obs.reshape(obs.shape[0], *obs.shape[2:])        

    def __repr__(self):
        return f'<{self.__class__.__name__}(idle_penalty={self.idle_penalty}, life_lost_penalty={self.life_lost_penalty}), {self.env}>'

## create_envs

In [13]:
def create_envs(envs_count, idle_penalty=None, life_lost_penalty=None, thread_pool_size=None, thread_affinity_offset=None, track_levels=False):
    assert 'NoFrameskip-v4' in HP.env.ident
    env_id = HP.env.ident[:HP.env.ident.index('NoFrameskip')].lower()
    frame_skip = 4
    
    envs = ale_py.AtariVectorEnv(
        game=env_id, 
        stack_num=1, # frame stacking is not needed because we use Transformer
        maxpool=True, # Combination of maxpool=True and 
        frameskip=frame_skip, # frameskip=4 corresponds to MaxAndSkipEnv(env, skip=4)
        repeat_action_probability=0, # corresponds NoFrameskip-v4
        use_fire_reset=True, # FireResetEnv(env)
        noop_max=30, # NoopResetEnv(env, noop_max=30)
        
        episodic_life=False, # will manage episodic life our selves
        life_loss_info=False, # strange parameter. When set to True then program will segfault
        reward_clipping=False, # will clip reward in wrapper in order to keep original rewards for game stats accounting

        # Get raw observation since rescaling and grayscaling is done on GPU
        grayscale=False,
        img_height=210,
        img_width=160,
        
        num_envs=envs_count, 
        num_threads=lu.coalesce(thread_pool_size, 0), # 0 means num of threads = num of envs
        thread_affinity_offset=lu.coalesce(thread_affinity_offset, -1), # -1 means no affinity

        autoreset_mode=gym.vector.vector_env.AutoresetMode.DISABLED,

        return_ram=track_levels,
    )

    return MyUberWrapper(
        envs, 
        idle_penalty=lu.coalesce(idle_penalty, HP.env.idle_penalty), 
        life_lost_penalty=lu.coalesce(life_lost_penalty, HP.env.life_lost_penalty),
        frame_skip=frame_skip, # used in video capturing to compute fps
        track_levels=track_levels,
    )

## Configure 

In [14]:
# @launchit.disable
# @launchit.collect
HP.env.ident = 'FrostbiteNoFrameskip-v4'
HP.env.count = 32 
HP.env.is_episodic_life = True
HP.env.actions_count = 6
HP.env.idle_penalty = 0
HP.env.life_lost_penalty = 0
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'system': {'comment': None,
            'random_seed': 42,
            'is_torch_deterministic': True,
            'is_torch_compile': True},
 'env': {'ident': 'FrostbiteNoFrameskip-v4',
         'count': 32,
         'is_episodic_life': True,
         'actions_count': 6,
         'idle_penalty': 0,
         'life_lost_penalty': 0},
 'agent': {'parent': None,
           'layers_count': 3,
           'heads_count': 4,
           'd_model': 256,
           'obs_sequence_length': 10,
           'action_plan_length': 10,
           'positional_encoding': 'learned'},
 'video': {'capture_policy': 'every(1000000)',
           'capture_preprocessed_obs': False,
           'capture_env_rams': None,
           'break_on_level_passed': None},
 'ppo': {'global_steps_count': 1000000,
         'rollout_steps_count': 512,
         'rollout_env_rams': None,
         'rollout_env_ram_patches': None,
         'tau': None,
       

## Create

In [15]:
# @launchit.disable_worker
envs = create_envs(1)
LOG(f'Test envs created: {envs}')
assert isinstance(envs.single_action_space, gym.spaces.Discrete), f'Only discrete action spaces are supported, but got {type(env.single_action_space)}'
assert envs.single_observation_space.shape == (1, 210, 160, 3)
LS.env_observation_space_shape = envs.single_observation_space.shape
LS.env_action_space_shape = (lu.coalesce(HP.env.actions_count, envs.single_action_space.n.item()),)
assert len(LS.env_action_space_shape) == 1

LOG(f'{envs.metadata=}')
LOG(f'{LS.env_observation_space_shape=}')
LOG(f'{LS.env_action_space_shape=}')

del envs

Test envs created: <MyUberWrapper(idle_penalty=0, life_lost_penalty=0), AtariVectorEnv(num_envs=1)>
envs.metadata={'autoreset_mode': <AutoresetMode.DISABLED: 'Disabled'>}
LS.env_observation_space_shape=(1, 210, 160, 3)
LS.env_action_space_shape=(6,)


# Agent

## Components

### PositionalEncoding

In [17]:
# dialogs/positional_embedding.ipynb
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, min_timescale=2.0, max_timescale=1e4):
        super().__init__()
        freqs = torch.arange(0, d_model, min_timescale) # e.g. -> [0, 2, 4, ..., 382], len(freqs) = d_model // min_timescale
        inv_freqs = max_timescale ** (-freqs / d_model) # e.g. [1, 0.96, 0.93, ... 0.001]
        self.register_buffer("inv_freqs", inv_freqs) # declare as non trainable parameter but still part of a model (e.g. included in state_dict, obeys to(device), etc.)

    def forward(self, seq_len):
        seq = torch.arange(seq_len - 1, -1, -1.0, device=self.inv_freqs.device) # -> [seq_len-1, seq_len-2, ... 0]
        sinusoidal_inp = einops.rearrange(seq, 'n -> n 1') * einops.rearrange(self.inv_freqs, 'd -> 1 d') # n -> n () == n -> n 1
        pos_emb = torch.cat((sinusoidal_inp.sin(), sinusoidal_inp.cos()), dim=-1)
        return pos_emb # [seq_len, d_model]

### PredictionHead

In [18]:
class PredictionHead(nn.Module):
    def __init__(self, actions_count, d_model, action_d_model = 64):
        super().__init__()
        self.action_d_model = action_d_model
        self.action_embedding = nn.Embedding(actions_count, self.action_d_model) 
        self.network = nn.Sequential(
            nn.Linear(d_model + self.action_d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

    def forward(self, transformer_outputs, actions):
        # transformer_outputs: [batch, (seq,)? d_model]
        # actions: [batch, (seq)?]
        act_emb = self.action_embedding(actions) # [batch, (seq,)? self.action_d_model]
        # Combine the history token with the action taken at that step
        combined = torch.cat([transformer_outputs, act_emb], dim=-1) # [batch, (seq,)? d_model + self.action_d_model]
        # Predict the next visual frame embedding
        return self.network(combined) # [batch, (seq,)? d_model]

## Agent

In [19]:
class Agent(nn.Module):
    @dataclass(slots=True)
    class Params:
        d_model: int = None
        layers_count: int = None
        heads_count: int = None
        positional_encoding: str = None
        observation_space_shape: tuple = (1, 84, 84)
        actions_count: int = None
        obs_sequence_length: int = None
        action_plan_length: int = None # how many actions to think of ahead
        
    def __init__(self, params):
        super().__init__()
        self.params = params
        assert self.params.obs_sequence_length >= 1
        assert self.params.action_plan_length >= 1
        self.sequence_length = self.params.obs_sequence_length + self.params.action_plan_length
        assert self.params.observation_space_shape == (1, 84, 84)
        input_color_channels = self.params.observation_space_shape[0]
        
        # @kms@ see hybrid model (GeLU, SiLU)?
        self.cnn = nn.Sequential(
            self._init_weights(nn.Conv2d(input_color_channels, 32, 8, stride=4)),
            nn.ReLU(),
            self._init_weights(nn.Conv2d(32, 64, 4, stride=2)),
            nn.ReLU(),
            self._init_weights(nn.Conv2d(64, 64, 3, stride=1)),
            nn.ReLU(),
            nn.Flatten(),
            self._init_weights(nn.Linear(64 * 7 * 7, self.params.d_model)), # collapse 64 features maps of 7x7 grid (49 elements) to a just single vector in embedding space
            nn.ReLU(),
        )
        
        assert params.positional_encoding in ['absolute', 'learned'], f'Unsupported {params.positional_encoding=}'
        
        if params.positional_encoding == 'absolute':
            self.pos_embedding = None # delay creation until frist forward. This is to create on target device
        elif params.positional_encoding == 'learned':
            self.pos_embedding = nn.Parameter(torch.randn(self.params.obs_sequence_length, params.d_model) * 0.02)

        self.causal_mask = None
        
        transformer_layer = nn.TransformerEncoderLayer(
            d_model=self.params.d_model, 
            dim_feedforward=self.params.d_model * 4, # kms@ default is 2048, 384 * 4 = 1536
            nhead=self.params.heads_count, 
            batch_first=True,
            norm_first=True,  # preferred for RL tasks
            dropout=0.0,      # Dropout destroys RL performance; keep 0.0
        )

        self.transformer = nn.TransformerEncoder(
            transformer_layer, 
            num_layers=self.params.layers_count,
            enable_nested_tensor=False, # True is incompatible with layers where norm_first=True
        )
        self._init_transformer_weights(self.transformer)

        self.action_plan_embs = nn.Parameter(torch.randn(self.params.action_plan_length, self.params.d_model))
        self.actor = self._init_weights(nn.Linear(self.params.d_model, out_features=self.params.actions_count), np.sqrt(0.01))
        self.critic = self._init_weights(nn.Linear(self.params.d_model, out_features=1), 1)
        self.predictor = PredictionHead(actions_count=self.params.actions_count, d_model=self.params.d_model) # kms@ init weights?

    def embed_obs(self, obs):
        # expected obs.shape = [batch, seq, color, height, width], i.e. batch with sequence of images in planar color layout
        obs_shape = obs.shape
        assert len(obs_shape) == 5, len(obs_shape)
        assert obs_shape[-3:] == (1, 84, 84), obs_shape
        obs = obs.view(-1, *obs_shape[-3:]) # [uberbatch, color, height, width]
        obs = self.cnn(obs) # [uberbatch, d_model]
        return obs.view(obs_shape[0], obs_shape[1], -1)  # [batch, seq, d_model]

    def generate_actions_noise(self, shape):
        device = self.action_plan_embs.device
        
        if isinstance(shape, tuple):
            u = torch.rand(*shape, self.params.actions_count, device=device)
        else:
            assert isinstance(shape, int)
            u = torch.rand(shape, self.params.actions_count, device=device)
            
        return -torch.log(-torch.log(u + 1e-10) + 1e-10)

    ForwardResult = namedtuple('ForwardResult', 'actions, action_log_probs, action_plan_logits, action_entropies, values, next_obs_embs')

    def forward(self, obs, padding_masks, actions_noise, tau, actions=None, predict_next_obs_emb=False):
        batch_size = len(obs)
        # call transformer, will internally extend obs with action_plan_embs
        embs = self._transform_obs(obs, padding_masks)  # Shape: [batch, total_seq_len, d_model]
        
        # Extract planned actions tokens -> [batch, action_plan_len, d_model]
        action_plan_embs = embs[:,-self.params.action_plan_length:] 
        assert action_plan_embs.ndim == 3
        
        # Map to logits -> [batch, action_plan_len, action_logits]
        action_plan_logits = self.actor(action_plan_embs) 
    
        # Scale logits by temperature for correct Gumbel-Max math alignment
        scaled_action_plan_logits = action_plan_logits / tau
    
        if actions is None:
            # --- Rollout Case ---
            actions_count = action_plan_logits.shape[-1]
            assert actions_noise.shape == (batch_size, actions_count)
            
            # Add noise to the TEMPERATURE-SCALED logits to emulate stochasticity (a-la Categorical.sample)
            noised_action_plan_logits = scaled_action_plan_logits[:,0] + actions_noise # [batch, action_logits]
            actions = torch.argmax(noised_action_plan_logits, dim=-1) # [batch]
            assert actions.shape == (batch_size,)
        else:
            # --- Optimization Case ---
            assert actions.shape == (batch_size,)
    
        # Compute stable log-probabilities 
        action_plan_log_probs = torch.log_softmax(scaled_action_plan_logits, dim=-1) # log_softmax == Categorical(logits=logits).log_prob(actions)
        batch_dim_inds = torch.arange(batch_size, device=action_plan_log_probs.device)
        action_log_probs = action_plan_log_probs[batch_dim_inds, 0, actions] 
        assert action_log_probs.shape == (batch_size,)
        
        # Evaluate entropies of whole action plans ("creative sandbox" branch)
        action_plan_probs = torch.softmax(scaled_action_plan_logits, dim=-1)
        action_plan_entropies = -torch.sum(action_plan_probs * action_plan_log_probs, dim=-1) # [batch, action_plan_len]
        
        current_state_embs = embs[:,(self.params.obs_sequence_length - 1)]
        
        return Agent.ForwardResult(
            actions=actions,
            action_log_probs=action_log_probs,
            action_plan_logits=action_plan_logits,
            action_entropies=action_plan_entropies,
            values=self.critic(current_state_embs).squeeze(-1), 
            next_obs_embs=lu.when(predict_next_obs_emb, lambda: self.predictor(current_state_embs, actions), None),
        )

    def _transform_obs(self, obs, padding_masks):
        # expected obs.shape = [batch, ob_seq, color, height, width], i.e. batch with sequence of images in planar color layout
        # obs is expected to be left-padded, i.e. the very fresh observation is [:,-1] within each batch (env) is the last one
        batch_size = len(obs)
        obs_embs = self.embed_obs(obs) # [batch, obs_seq, d_model]

        if self.pos_embedding is None:
            assert self.params.positional_encoding == 'absolute'
            self.pos_embedding = PositionalEncoding(self.params.d_model)(self.params.obs_sequence_length)
            self.pos_embedding = self.pos_embedding.to(obs_embs.device)

        assert self.pos_embedding.shape == (self.params.obs_sequence_length, self.params.d_model)
        obs_embs = obs_embs + self.pos_embedding.unsqueeze(0)
        
        action_plan_embs = self.action_plan_embs.unsqueeze(0).expand(batch_size, -1, -1)
        full_embs = torch.cat([obs_embs, action_plan_embs], dim=1) # [batch, seq, d_model]

        if self.causal_mask is None:
            self.causal_mask = nn.Transformer.generate_square_subsequent_mask(self.sequence_length)
            self.causal_mask = self.causal_mask.to(full_embs.device)

        action_plan_padding_masks = torch.full((batch_size, self.params.action_plan_length), 0, dtype=padding_masks.dtype, device=padding_masks.device)
        full_padding_masks = torch.cat([padding_masks, action_plan_padding_masks], dim=1) # [batch,seq]
        assert full_padding_masks.shape == (batch_size, self.sequence_length)

        return self.transformer(
            full_embs, 
            src_key_padding_mask=full_padding_masks,
            mask=self.causal_mask, 
            is_causal=True)

    @staticmethod
    def preprocess_obs(obs):
        assert isinstance(obs, torch.Tensor)
        obs_ndim = obs.ndim
        assert obs_ndim >= 3 # height, width, color

        if obs_ndim == 3:
            obs = obs.unsqueeze(0)
        
        obs_shape = obs.shape
        obs = obs.view(-1, *obs_shape[-3:])
        obs = obs.permute(0, 3, 1, 2) # move color channel to the front (switch interleaved->planar format)

        obs = VF.rgb_to_grayscale(obs, num_output_channels=1)
        obs = VF.resize(obs, [84, 84], interpolation=VF.InterpolationMode.BILINEAR, antialias=True)
        
        obs = obs.float() / 255.0
        obs = obs.view(*obs_shape[:-3], 1, 84, 84)

        if obs_ndim == 3:
            obs = obs.squeeze(0)
        
        return obs

    @staticmethod
    def _init_weights(l, gain=np.sqrt(2), bias_const=0.0):
        '''
        CleanRL style orthogonal initialization helper
        https://share.google/aimode/BDty0f9ZJZ6OXBFFQ
        '''
        
        nn.init.orthogonal_(l.weight, gain=gain)
        
        if l.bias is not None:
            nn.init.constant_(l.bias, bias_const)
            
        return l

    @staticmethod
    def _init_transformer_weights(t):
        '''
        Applies orthogonal initialization to the inner transformer blocks.
        https://share.google/aimode/BDty0f9ZJZ6OXBFFQ
        '''
        
        for name, param in t.named_parameters():
            if 'weight' in name and param.dim() >= 2:
                # Appling gain=1.0 keeps variance stable across depth
                nn.init.orthogonal_(param, gain=1.0)
            elif 'bias' in name:
                nn.init.constant_(param, 0.0)

## Test

### Basics

In [64]:
# @launchit.disable
t = lu.ScopedVars()
t.device = CONFIG.cuda_device
# t.device = 'cpu'

t.ap = Agent.Params(
    d_model=384,
    layers_count=3,
    heads_count=4,
    positional_encoding='learned',
    actions_count=18,
    obs_sequence_length=10,
    action_plan_length=10,
)
t.agent = Agent(t.ap).to(t.device)
print(t.agent)
params_count = sum(p.numel() for p in t.agent.parameters())
print(f'{params_count=:_}')

t.envs_count = 2
t.steps_count = 10

t.obs_bufs = torch.zeros((t.envs_count, t.ap.obs_sequence_length, *t.ap.observation_space_shape)).to(t.device)
t.obs_pmasks = torch.ones((t.envs_count, t.ap.obs_sequence_length)).to(t.device)
t.obs_pmasks[:,-1] = 0

print(f'{t.obs_bufs.shape=}')
print(f'{t.obs_pmasks.shape=}')

r = t.agent(
    obs=t.obs_bufs,
    padding_masks=t.obs_pmasks,
    actions_noise=t.agent.generate_actions_noise(t.envs_count),
    tau=0.1,
)

shape = einops.parse_shape(r.actions, 'e')
assert shape['e'] == t.envs_count
print(f'{r.actions.shape=}, {shape=}')

shape = einops.parse_shape(r.action_log_probs, 'e')
assert shape['e'] == t.envs_count
print(f'{r.action_log_probs.shape=}, {shape=}')

shape = einops.parse_shape(r.action_plan_logits, 'e a l')
assert shape['e'] == t.envs_count
assert shape['a'] == t.ap.action_plan_length
assert shape['l'] == t.ap.actions_count
print(f'{r.action_plan_logits.shape=}, {shape=}')

shape = einops.parse_shape(r.action_entropies, 'e a')
assert shape['e'] == t.envs_count
assert shape['a'] == t.ap.action_plan_length
print(f'{r.action_entropies.shape=}, {shape=}')

shape = einops.parse_shape(r.values, 'e')
assert shape['e'] == t.envs_count
print(f'{r.values.shape=}, {shape=}')

Agent(
  (cnn): Sequential(
    (0): Conv2d(1, 32, kernel_size=(8, 8), stride=(4, 4))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (5): ReLU()
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=3136, out_features=384, bias=True)
    (8): ReLU()
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=384, out_features=384, bias=True)
        )
        (linear1): Linear(in_features=384, out_features=1536, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear2): Linear(in_features=1536, out_features=384, bias=True)
        (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
        

### Quick play

In [72]:
# @launchit.disable
t = lu.ScopedVars()
t.sequence_length = 10
t.device = CONFIG.cuda_device
# t.device = 'cpu'

t.ap = Agent.Params(
    d_model=256,
    layers_count=3,
    heads_count=4,
    positional_encoding='absolute',
    actions_count=LS.env_action_space_shape[0],
    action_plan_length=10,
    obs_sequence_length=t.sequence_length,
)
t.agent = Agent(t.ap).to(t.device)

t.env_step = 0
t.env = create_envs(1, idle_penalty=-0.01, life_lost_penalty=-3)
t.obs, _ = t.env.reset(seed=HP.system.random_seed)
t.obs = torch.tensor(t.obs).to(t.device) 
t.obs = Agent.preprocess_obs(t.obs.squeeze(0))
t.obs_buf = torch.zeros((t.sequence_length, *t.ap.observation_space_shape)).to(t.device)
t.obs_buf[-1] = t.obs
t.obs_pmask = torch.ones(t.sequence_length).to(t.device)
t.obs_pmask[-1] = 0
t.rewards_sum = 0

t.rollout_steps_count = 1000

# verification vars
t.vrf_obs_buf = deque(maxlen=t.sequence_length)
t.vrf_obs_buf.append(t.obs)

t.vrf_obs_pmasks = []

for i in range(t.rollout_steps_count):
    t.vrf_obs_pmask = torch.ones(t.sequence_length)
    t.vrf_obs_pmask[-(i+1):] = 0
    t.vrf_obs_pmasks.append(t.vrf_obs_pmask)

t.vrf_obs_pmasks = torch.vstack(t.vrf_obs_pmasks).to(t.device)

with torch.no_grad():
    for t.rs in tqdm(range(0, t.rollout_steps_count)): 
        # verification of FIFO logic correcteness
        t.vrf_obs_buf_as_tensor = einops.rearrange(torch.vstack(list(t.vrf_obs_buf)), '(a b) ...-> a b ...', a=len(t.vrf_obs_buf))
        assert torch.all(t.obs_buf[~t.obs_pmask.bool()] == t.vrf_obs_buf_as_tensor)
        assert torch.all(t.obs_pmask == t.vrf_obs_pmasks[t.env_step])

        # Each call takes 2-3 ms with GPU, so throughput is limited to 1000/2.5 = 400 it/s
        t.agent_result = t.agent(
            obs=t.obs_buf.unsqueeze(0),
            padding_masks=t.obs_pmask.unsqueeze(0),
            actions_noise=t.agent.generate_actions_noise(1),
            tau=0.1,
        )

        t.action = t.agent_result.actions[0].cpu().unsqueeze(0).numpy()
            
        t.obs, t.reward, t.terminated, t.truncated, t.info = t.env.step(t.action)
        t.obs = torch.tensor(t.obs.squeeze(0)).to(t.device) # squeeze(1) - squeeze dummy stack frame dim, squeeze(0) - squeeze dummy env dim
        t.obs = Agent.preprocess_obs(t.obs)
        
        # FIFO logic - push obs into tail, stale would fall out (and forget) from the head
        t.obs_buf = t.obs_buf.roll(shifts=-1, dims=0)
        t.obs_buf[-1] = t.obs
        
        t.obs_pmask = t.obs_pmask.roll(shifts=-1, dims=0)
        t.obs_pmask[-1] = 0

        t.rewards_sum += t.reward 

        t.vrf_obs_buf.append(t.obs)

        if t.info['is_life_lost'][0]:
            t.r = t.info['life_returns'][0].item()
            t.l = t.info['life_lengths'][0].item()
            LOG(f'{t.env_step:03} Life lost: r={t.r:6}, l={t.l:6}, rs={t.rewards_sum}')
            # LOG(f'{t.env_step:03} Life lost: r={t.r:6}, l={t.l:6}; {t.info=}')

        if t.info['is_episode_over'][0]:
            t.r = t.info['episode_returns'][0].item()
            t.l = t.info['episode_lengths'][0].item()
            LOG(f'{t.env_step:03} EPISODE OVER: r={t.r:6}, l={t.l:6}, rs={t.rewards_sum}')
            # LOG(f'{t.env_step:03} EPISODE OVER: r={t.r:6}, l={t.l:6}; {t.info=}')

        if t.terminated or t.truncated:
            t.obs, _ = t.env.reset()
            t.obs = torch.tensor(t.obs.squeeze(0)).to(t.device)
            t.obs = Agent.preprocess_obs(t.obs)
            
            t.obs_buf.zero_()
            t.obs_buf[-1] = t.obs
            t.obs_pmask.fill_(1)
            t.obs_pmask[-1] = 0
            
            t.vrf_obs_buf.clear()
            t.vrf_obs_buf.append(t.obs)
            
            t.env_step = 0
        else:
            assert not 'episode' in t.info
            assert not 'is_game_over' in t.info
            t.env_step += 1

  0%|          | 0/1000 [00:00<?, ?it/s]

072 Life lost: r=   0.0, l=   303, rs=[-3.72]
147 Life lost: r=  10.0, l=   300, rs=[-6.45]
222 Life lost: r=   0.0, l=   300, rs=[-10.19]
245 Life lost: r=   0.0, l=    92, rs=[-13.41]
245 EPISODE OVER: r=  10.0, l=   995, rs=[-13.41]
073 Life lost: r=   0.0, l=   319, rs=[-17.14]
149 Life lost: r=   0.0, l=   304, rs=[-20.89]
229 Life lost: r=   0.0, l=   320, rs=[-24.68]
255 Life lost: r=   0.0, l=   102, rs=[-27.93]
255 EPISODE OVER: r=   0.0, l=  1045, rs=[-27.93]
131 Life lost: r=  10.0, l=   556, rs=[-31.23]
205 Life lost: r=   0.0, l=   296, rs=[-34.96]
382 Life lost: r=  30.0, l=   708, rs=[-36.69]
405 Life lost: r=   0.0, l=    92, rs=[-39.91]
405 EPISODE OVER: r=  40.0, l=  1652, rs=[-39.91]


### VectorEnv

In [78]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = 2
t.sequence_length = 10
t.device = CONFIG.cuda_device
# t.device = 'cpu'

t.ap = Agent.Params(
    d_model=256,
    layers_count=3,
    heads_count=4,
    positional_encoding='absolute',
    actions_count=LS.env_action_space_shape[0],
    action_plan_length=10,
    obs_sequence_length=t.sequence_length,
)
t.agent = Agent(t.ap).to(t.device)

t.env_steps = torch.zeros(t.envs_count).long()
t.envs = create_envs(t.envs_count, idle_penalty=-0.01, life_lost_penalty=-3)
t.obs, _ = t.envs.reset(seed=HP.system.random_seed)
t.obs = torch.tensor(t.obs).to(t.device)
t.obs = Agent.preprocess_obs(t.obs)
t.obs_bufs = torch.zeros((t.envs_count, t.sequence_length, *t.ap.observation_space_shape)).to(t.device)
t.obs_bufs[:,-1] = t.obs
t.obs_pmasks = torch.ones((t.envs_count, t.sequence_length)).to(t.device)
t.obs_pmasks[:,-1] = 0
t.rewards_sums = np.zeros(t.envs_count)

t.rollout_steps_count = 1000
t.life_lost_r_sum = 0
t.episode_over_r_sum = 0

with torch.no_grad():
    for t.rs in tqdm(range(0, t.rollout_steps_count)): 
        # Each call takes 2-3 ms with GPU, so throughput is limited to 1000/2.5 = 400 it/s
        t.agent_result = t.agent(
            obs=t.obs_bufs,
            padding_masks=t.obs_pmasks,
            actions_noise=t.agent.generate_actions_noise(t.envs_count),
            tau=0.1,
        )

        t.obs, t.rewards, t.terminations, t.truncations, t.infos = t.envs.step(t.agent_result.actions.cpu().numpy())
        t.obs = torch.tensor(t.obs).to(t.device)
        t.obs = Agent.preprocess_obs(t.obs)
        
        # FIFO logic - push obs into tail, stale would fall out (and forget) from the head
        t.obs_bufs = t.obs_bufs.roll(shifts=-1, dims=1)
        t.obs_bufs[:,-1] = t.obs
        
        t.obs_pmasks = t.obs_pmasks.roll(shifts=-1, dims=1)
        t.obs_pmasks[:,-1] = 0

        assert t.rewards_sums.shape == t.rewards.shape
        t.rewards_sums += t.rewards

        if 'is_life_lost' in t.infos:
            for t.env_ind in np.argwhere(t.infos['is_life_lost']).ravel():
                t.r = t.infos['life_returns'][t.env_ind]
                t.l = t.infos['life_lengths'][t.env_ind]
                t.rs = t.rewards_sums[t.env_ind]
                LOG(f'{t.env_steps[t.env_ind]:03} ENV:{t.env_ind} Life lost: r={t.r:6}, l={t.l:6}, rs={t.rs:.2f}, lives={t.infos['lives'][t.env_ind]}')
                t.life_lost_r_sum += t.r

        if 'is_episode_over' in t.infos:
            for t.env_ind in np.argwhere(t.infos['is_episode_over']).ravel():
                t.r = t.infos['episode_returns'][t.env_ind]
                t.l = t.infos['episode_lengths'][t.env_ind]
                t.rs = t.rewards_sums[t.env_ind]
                LOG(f'{t.env_steps[t.env_ind]:03} ENV:{t.env_ind} EPISOVE OVER: r={t.r:6}, l={t.l:6}, rs={t.rs:.2f}, lives={t.infos['lives'][t.env_ind]}')
                t.episode_over_r_sum += t.r

        t.done_envs = np.logical_or(t.terminations, t.truncations)
            
        for t.env_ind, t.is_done in enumerate(t.done_envs):
            if not t.is_done:
                t.env_steps[t.env_ind] += 1
            else:
                # t.obs[t.env_ind] is a death screen, handle it properly, e.g. preserve in a rollout buffer
                t.obs_pmasks[t.env_ind] = 1
                t.obs_pmasks[t.env_ind,-1] = 0
                t.env_steps[t.env_ind] = 0

        if np.any(t.done_envs):
            t.reset_obs, t.reset_infos = t.envs.reset(options=dict(reset_mask=t.done_envs))
            LOG(f'{t.done_envs=}, {t.reset_infos=}')

            for t.env_ind in np.argwhere(t.done_envs).ravel():
                assert t.reset_infos['lives'][t.env_ind] > t.infos['lives'][t.env_ind]
                assert t.reset_infos['episode_frame_number'][t.env_ind] <= t.infos['episode_frame_number'][t.env_ind]

            for t.env_ind in np.argwhere(~t.done_envs).ravel():
                assert t.reset_infos['lives'][t.env_ind] == t.infos['lives'][t.env_ind]
                assert t.reset_infos['episode_frame_number'][t.env_ind] == t.infos['episode_frame_number'][t.env_ind]
            
            t.reset_obs = torch.tensor(t.reset_obs[t.done_envs]).to(t.device)
            t.reset_obs = Agent.preprocess_obs(t.reset_obs)
            t.old_obs = t.obs[~t.done_envs].clone()
            t.obs[t.done_envs] = t.reset_obs
            assert torch.all(t.old_obs == t.obs[~t.done_envs]) # ensure we didn't touch envs which are not done

print(f'{t.life_lost_r_sum=}, {t.episode_over_r_sum=}')

  0%|          | 0/1000 [00:00<?, ?it/s]

074 ENV:0 Life lost: r=   0.0, l=   311, rs=0.00, lives=3
079 ENV:1 Life lost: r=   0.0, l=   323, rs=0.00, lives=3
149 ENV:0 Life lost: r=   0.0, l=   300, rs=0.00, lives=2
158 ENV:1 Life lost: r=   0.0, l=   316, rs=0.00, lives=2
224 ENV:0 Life lost: r=   0.0, l=   300, rs=0.00, lives=1
235 ENV:1 Life lost: r=   0.0, l=   308, rs=0.00, lives=1
250 ENV:0 Life lost: r=   0.0, l=   102, rs=0.00, lives=0
250 ENV:0 EPISOVE OVER: r=   0.0, l=  1013, rs=0.00, lives=0
t.done_envs=array([ True, False]), t.reset_infos={'env_id': array([0, 1], dtype=int32), 'lives': array([4, 1], dtype=int32), 'frame_number': array([1036, 1007], dtype=int32), 'episode_frame_number': array([  23, 1007], dtype=int32)}
262 ENV:1 Life lost: r=   0.0, l=   106, rs=0.00, lives=0
262 ENV:1 EPISOVE OVER: r=   0.0, l=  1053, rs=0.00, lives=0
t.done_envs=array([False,  True]), t.reset_infos={'env_id': array([0, 1], dtype=int32), 'lives': array([4, 4], dtype=int32), 'frame_number': array([1084, 1067], dtype=int32), 'episo

### Batched FIFO logic

In [ ]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = 2
t.sequence_length = 10
t.obs_shape = (84, 84, 3)
t.rollout_steps_count = 100

t.env_steps = torch.zeros(t.envs_count, dtype=torch.long)
t.env_inds = torch.arange(t.envs_count)
t.obs_bufs = torch.zeros((t.envs_count, t.sequence_length, *t.obs_shape))
t.obs_pmasks = torch.ones((t.envs_count, t.sequence_length))

# Verification vars
t.vrf_obs_bufs = torch.zeros((t.envs_count, t.sequence_length, *t.obs_shape))
t.vrf_obs_pmasks = torch.ones((t.envs_count, t.sequence_length))

for i in tqdm(range(t.rollout_steps_count)):
    t.new_obs = torch.rand((t.envs_count, *t.obs_shape))

    # Serial version of FIFO logic
    for j in range(t.envs_count):
        t.vrf_obs_bufs[j] = t.vrf_obs_bufs[j].roll(shifts=-1, dims=0)
        t.vrf_obs_bufs[j,-1] = t.new_obs[j]
        
        t.vrf_obs_pmasks[j] = t.vrf_obs_pmasks[j].roll(shifts=-1, dims=0)
        t.vrf_obs_pmasks[j,-1] = 0

    # Batched version of FIFO logic
    t.obs_bufs = t.obs_bufs.roll(shifts=-1, dims=1)
    t.obs_bufs[:,-1] = t.new_obs

    t.obs_pmasks = t.obs_pmasks.roll(shifts=-1, dims=1)
    t.obs_pmasks[:,-1] = 0

    # Serial and batched must produce the same results
    assert torch.all(t.obs_bufs == t.vrf_obs_bufs)
    assert torch.all(t.obs_pmasks == t.vrf_obs_pmasks)

    for t.env_ind, t.done in enumerate(torch.rand(t.envs_count) < 0.05):
        if t.done:
            t.obs_bufs[t.env_ind].zero_()
            t.obs_pmasks[t.env_ind].fill_(1)

            t.vrf_obs_bufs[t.env_ind].zero_()
            t.vrf_obs_pmasks[t.env_ind].fill_(1)
            LOG(f'Rollout step {i}, env {t.env_ind} game over, restarting')

    t.env_steps += 1

### Obs reconstruction

In [36]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = 2
t.sequence_length = 10
t.obs_shape = (84, 84, 3)
t.rollout_steps_count = 100

t.env_inds = torch.arange(t.envs_count)
t.obs_storage = torch.zeros((t.envs_count, t.rollout_steps_count + 1, *t.obs_shape))
t.obs_bufs = torch.zeros((t.envs_count, t.sequence_length, *t.obs_shape))
t.obs_inds = torch.zeros((t.envs_count, t.sequence_length)).long()

for t.rollout_step in tqdm(range(t.rollout_steps_count)):
    t.new_obs = torch.rand((t.envs_count, *t.obs_shape))

    t.obs_storage[:,t.rollout_step+1] = t.new_obs

    t.obs_bufs = t.obs_bufs.roll(shifts=-1, dims=1)
    t.obs_inds = t.obs_inds.roll(shifts=-1, dims=1)
    
    t.obs_bufs[:,-1] = t.new_obs
    t.obs_inds[:,-1] = t.rollout_step + 1

    t.dones = torch.rand(2) < 0.05

    for t.env_ind, t.done in enumerate(t.dones):
        if t.done:
            t.obs_bufs[t.env_ind].zero_()
            t.obs_inds[t.env_ind].zero_()

    t.obs_inds_ex = einops.rearrange(t.obs_inds, 'e m -> e m 1 1 1')
    t.obs_inds_ex = t.obs_inds_ex.expand((-1, -1, *t.obs_storage.shape[2:]))
    t.reconstr_obs_bufs = torch.gather(t.obs_storage, dim=1, index=t.obs_inds_ex)

    assert torch.all(t.obs_bufs == t.reconstr_obs_bufs)

  0%|          | 0/100 [00:00<?, ?it/s]

## Configure

In [20]:
# @launchit.disable
# @launchit.collect
HP.agent.parent = None # e.g. 17e_ppo_tr_atari_mp_10:3
HP.agent.layers_count = 3 # number of transformer layers
HP.agent.heads_count = 4 # number of heads used in multi-head attention
HP.agent.d_model = 256 # dimension of the transformer
HP.agent.obs_sequence_length = 4 # length observation chain agent incepts
HP.agent.action_plan_length = 10 # number of actions agent must think upfront about
HP.agent.positional_encoding = 'learned' # positional encoding type of the transformer: "", "absolute", "learned"
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'system': {'comment': None,
            'random_seed': 42,
            'is_torch_deterministic': True,
            'is_torch_compile': True},
 'env': {'ident': 'FrostbiteNoFrameskip-v4',
         'count': 32,
         'is_episodic_life': True,
         'actions_count': 6,
         'idle_penalty': 0,
         'life_lost_penalty': 0},
 'agent': {'parent': None,
           'layers_count': 3,
           'heads_count': 4,
           'd_model': 256,
           'obs_sequence_length': 4,
           'action_plan_length': 10,
           'positional_encoding': 'learned'},
 'video': {'capture_policy': 'every(1000000)',
           'capture_preprocessed_obs': False,
           'capture_env_rams': None,
           'break_on_level_passed': None},
 'ppo': {'global_steps_count': 1000000,
         'rollout_steps_count': 512,
         'rollout_env_rams': None,
         'tau': None,
         'epochs_count': 3,
         'minibatches_

## Initrd

In [21]:
# @launchit.collect_initrd
# @launchit.disable
if CONFIG.exec_mode in [ExecMode.MASTER_NOTEBOOK, ExecMode.LAUNCH_NOTEBOOK] and HP.agent.parent is not None:
    assert os.path.exists(CONFIG.initrd_path)
    assert CONFIG.model_group_uri is not None
    
    parent = hp_parse_artifact_source(HP.agent.parent)
    local_artifact_registry = ArtifactRegistry(CONFIG.model_group_uri)
    pt_data = local_artifact_registry.get_asset_content(parent.model_name, parent.model_version, asset_classifier='agent', asset_ext='pt')
    fname = os.path.join(CONFIG.initrd_path, 'agent.pt')
    
    with open(fname, 'wb') as f:
        f.write(pt_data)    
        Logging.get()(f'agent.pt saved to "{fname}"')

    agent_params = local_artifact_registry.get_asset_content(parent.model_name, parent.model_version, asset_classifier='agent_params', asset_ext='json')
    fname = os.path.join(CONFIG.initrd_path, 'agent_params.json')

    with open(fname, 'wt') as f:
        f.write(agent_params.decode('utf-8'))
        Logging.get()(f'agent_params.json saved to "{fname}"')

#launchit.stop

## Create

In [22]:
# @launchit.disable_worker
ap = Agent.Params(
    d_model=HP.agent.d_model,
    layers_count=HP.agent.layers_count,
    heads_count=HP.agent.heads_count,
    positional_encoding=HP.agent.positional_encoding,
    actions_count=LS.env_action_space_shape[0],
    obs_sequence_length=HP.agent.obs_sequence_length,
    action_plan_length=HP.agent.action_plan_length,
)
LS.agent = Agent(ap).to(CONFIG.cuda_device)
LS.agent.share_memory()
LOG(f'Agent created')

if HP.system.is_torch_compile:
    LS.agent = torch.compile(LS.agent, fullgraph=True)
    LOG(f'Agent compiled')

if HP.agent.parent is not None:
    assert os.path.exists(CONFIG.initrd_path)
    fname = os.path.join(CONFIG.initrd_path, 'agent_params.json')

    with open(fname, 'rt') as f:
        agent_params = Agent.Params(**json.load(f))
        agent_params.observation_space_shape = tuple(agent_params.observation_space_shape)
        assert agent_params == LS.agent.params

    fname = os.path.join(CONFIG.initrd_path, 'agent.pt')
    
    with open(fname, 'rb') as f:
        LS.agent.load_state_dict(torch.load(f))

    LOG(f'Agent state loaded from "{fname}"')

Agent created
Agent compiled


# Dataset

## Dataset

In [ ]:
@dataclass(slots=True)
class Dataset:
    # Metadata
    obs_sequence_length: int = None
    prologue_size: int = None
    
    # Agent's input data
    obs: object = None
    obs_pmasks: object = None # padding masks
    
    # Agent's output data
    actions: object = None
    action_log_probs: object = None
    values: object = None
    next_action_plan_logits: object = None
    
    # Environment response
    next_obs: object = None # observation which agent received after stepping (used to learn predictions)
    dones: object = None
    advantages: object = None
    returns: object = None
    
    # Debug
    debug_w_obs: object = None
    debug_action_plan_logits: object = None
    debug_lookahead_action_plan_logits: object = None

    # IPC tribute
    shared_tensors: dict = None
    
    # Iteration support
    valid_inds: object = None
    batch_size: int = None
    is_shuffled: bool = None
    iter_inds: object = None
    iter_pos: int = None
    flattened: tuple = None
    selected: tuple = None
    bw_inds_stem: object = None

    Batch = namedtuple('Batch', 'obs, obs_pmasks, actions, action_log_probs, values, next_action_plan_logits, next_obs, dones, advantages, returns, b_inds')
    
    def __init__(
        self,
        envs_count, 
        observation_space_shape, 
        rollout_steps_count, 
        obs_sequence_length,
        action_plan_length,
        actions_count,
        batches_count,
        is_shuffled,
        is_next_obs,
        is_debug=False,
    ):
        self.obs_sequence_length = obs_sequence_length
        self.prologue_size = (obs_sequence_length - 1)
        rollout_buf_size = self.prologue_size + rollout_steps_count # prologue (mem window from prev rollout) + actual rollout data
        assert rollout_buf_size > 0
        
        # Agent's input data
        self.obs = torch.zeros((envs_count, rollout_buf_size, *observation_space_shape)).to(CONFIG.cuda_device)
        self.obs_pmasks = torch.zeros((envs_count, rollout_buf_size, obs_sequence_length)).to(CONFIG.cuda_device) 

        # Agent's output data
        self.actions = torch.zeros((envs_count, rollout_buf_size), dtype=torch.long).to(CONFIG.cuda_device)
        self.action_log_probs = torch.zeros((envs_count, rollout_buf_size)).to(CONFIG.cuda_device)
        self.values = torch.zeros((envs_count, rollout_buf_size)).to(CONFIG.cuda_device)
        self.next_action_plan_logits = torch.zeros((envs_count, rollout_buf_size, action_plan_length, actions_count)).to(CONFIG.cuda_device)

        # Environment response
        self.next_obs = lu.when(is_next_obs, lambda: torch.zeros((envs_count, rollout_buf_size, *observation_space_shape)).to(CONFIG.cuda_device), None)
        self.dones = torch.zeros((envs_count, rollout_buf_size)).to(CONFIG.cuda_device)
        self.advantages = torch.zeros((envs_count, rollout_buf_size)).to(CONFIG.cuda_device)
        self.returns = torch.zeros((envs_count, rollout_buf_size)).to(CONFIG.cuda_device)

        # Debug
        self.debug_w_obs = lu.when(is_debug, lambda: torch.zeros((envs_count, rollout_buf_size, obs_sequence_length, *observation_space_shape)).to(CONFIG.cuda_device), None)
        self.debug_action_plan_logits = lu.when(is_debug, lambda: torch.zeros((envs_count, rollout_buf_size, action_plan_length, actions_count)).to(CONFIG.cuda_device), None)
        self.debug_lookahead_action_plan_logits = lu.when(is_debug, lambda: torch.zeros((envs_count, 1, action_plan_length, actions_count)).to(CONFIG.cuda_device), None)
    
        self.shared_tensors = dict(
            obs=self.obs,
            obs_pmasks=self.obs_pmasks,
            actions=self.actions,
            action_log_probs=self.action_log_probs,
            values=self.values,
            next_action_plan_logits=self.next_action_plan_logits,
            next_obs=self.next_obs,
            dones=self.dones,
            advantages=self.advantages,
            returns=self.returns,
            debug_w_obs=self.debug_w_obs,
            debug_action_plan_logits=self.debug_action_plan_logits,
            debug_lookahead_action_plan_logits=self.debug_lookahead_action_plan_logits,
        )
        
        for t in self.shared_tensors.values():
            if t is not None:
                t.share_memory_()

        # Iteration support
        valid_inds = []
        
        for env_ind in range(envs_count):
            offset = env_ind * rollout_buf_size + self.prologue_size
            indices_for_env = offset + torch.arange(rollout_steps_count)
            valid_inds.append(indices_for_env)
            
        self.valid_inds = torch.cat(valid_inds).to(CONFIG.cuda_device)
        assert envs_count * rollout_steps_count % batches_count == 0, f'Not whole number of batches for given rollout data size: {((envs_count * rollout_steps_count) / batches_count)=}'
        self.batch_size = envs_count * rollout_steps_count // batches_count
        self.is_shuffled = is_shuffled

        self.flattened = Dataset.Batch(
            obs=self.obs.view(-1, *self.obs.shape[2:]),
            obs_pmasks=self.obs_pmasks.view(-1, *self.obs_pmasks.shape[2:]),
            actions=self.actions.view(-1, *self.actions.shape[2:]),
            action_log_probs=self.action_log_probs.view(-1, *self.action_log_probs.shape[2:]),
            values=self.values.view(-1, *self.values.shape[2:]),
            next_action_plan_logits=self.next_action_plan_logits.view(-1, *self.next_action_plan_logits.shape[2:]),
            next_obs=lu.when(is_next_obs, lambda: self.next_obs.view(-1, *self.next_obs.shape[2:]), None),
            dones=self.dones.view(-1, *self.dones.shape[2:]),
            advantages=self.advantages.view(-1, *self.advantages.shape[2:]),
            returns=self.returns.view(-1, *self.returns.shape[2:]),
            b_inds=None,
        )
        self.selected = Dataset.Batch(
            obs=torch.zeros((self.batch_size * obs_sequence_length, *observation_space_shape)).to(CONFIG.cuda_device),
            obs_pmasks=torch.ones((self.batch_size, obs_sequence_length)).to(CONFIG.cuda_device), # obs_pmasks is the only shared tensor which is stored as is
            actions=torch.zeros(self.batch_size, dtype=torch.long).to(CONFIG.cuda_device),
            action_log_probs=torch.zeros(self.batch_size).to(CONFIG.cuda_device),
            values=torch.zeros(self.batch_size).to(CONFIG.cuda_device),
            next_action_plan_logits=torch.zeros(self.batch_size, action_plan_length, actions_count).to(CONFIG.cuda_device),
            next_obs=lu.when(is_next_obs, lambda: torch.zeros((self.batch_size, *observation_space_shape)).to(CONFIG.cuda_device), None),
            dones=torch.zeros(self.batch_size).to(CONFIG.cuda_device),
            advantages=torch.zeros(self.batch_size).to(CONFIG.cuda_device),
            returns=torch.zeros(self.batch_size).to(CONFIG.cuda_device),
            b_inds=None,
        )
        self.bw_inds_stem = torch.arange(-obs_sequence_length + 1, 1).to(CONFIG.cuda_device)
        self.bw_inds_stem = self.bw_inds_stem.unsqueeze(0).expand(self.batch_size, self.bw_inds_stem.shape[0])

    def __iter__(self):
        if self.is_shuffled:
            valids_inds_order = torch.randperm(len(self.valid_inds), device=CONFIG.cuda_device)
            self.iter_inds = self.valid_inds[valids_inds_order]
        else:
            self.iter_inds = self.valid_inds

        self.iter_pos = 0
        return self
    
    def __next__(self):
        if self.iter_pos >= len(self.iter_inds):
            raise StopIteration

        b_inds = self.iter_inds[self.iter_pos:self.iter_pos+self.batch_size]
        self.iter_pos += self.batch_size
        
        torch.gather(input=self.flattened.obs_pmasks, index=b_inds.unsqueeze(1).expand((-1, self.obs_sequence_length)), dim=0, out=self.selected.obs_pmasks)
        
        bw_inds = self.bw_inds_stem + b_inds.unsqueeze(1) # batch-window indices, shape: [batch, seq_len]
        bw_inds = bw_inds.view(-1)
        
        torch.index_select(input=self.flattened.obs, index=bw_inds, dim=0, out=self.selected.obs)
        torch.index_select(input=self.flattened.actions, index=b_inds, dim=0, out=self.selected.actions)
        torch.index_select(input=self.flattened.action_log_probs, index=b_inds, dim=0, out=self.selected.action_log_probs)
        torch.index_select(input=self.flattened.values, index=b_inds, dim=0, out=self.selected.values)
        torch.index_select(input=self.flattened.next_action_plan_logits, index=b_inds, dim=0, out=self.selected.next_action_plan_logits)
        torch.index_select(input=self.flattened.dones, index=b_inds, dim=0, out=self.selected.dones)
        torch.index_select(input=self.flattened.advantages, index=b_inds, dim=0, out=self.selected.advantages)
        torch.index_select(input=self.flattened.returns, index=b_inds, dim=0, out=self.selected.returns)

        if self.next_obs is not None:
            torch.index_select(input=self.flattened.next_obs, index=b_inds, dim=0, out=self.selected.next_obs)
            next_obs = self.selected.next_obs
        else:
            next_obs = None
        
        return Dataset.Batch(
            obs=self.selected.obs.view((self.batch_size, self.obs_sequence_length, *self.selected.obs.shape[1:])),
            obs_pmasks=self.selected.obs_pmasks,
            actions=self.selected.actions,
            action_log_probs=self.selected.action_log_probs,
            values=self.selected.values,
            next_action_plan_logits=self.selected.next_action_plan_logits.view((self.batch_size, *self.next_action_plan_logits.shape[2:])),
            next_obs=next_obs,
            dones=self.selected.dones,
            advantages=self.selected.advantages,
            returns=self.selected.returns,
            b_inds=b_inds,
        )

## Test

### Smoke test

In [ ]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = 8
t.rollout_steps_count = 64
t.obs_sequence_length = 10
t.action_plan_length = 8
t.actions_count = 6
t.dataset = Dataset(
    envs_count=t.envs_count,
    observation_space_shape=LS.agent.params.observation_space_shape, 
    rollout_steps_count=t.rollout_steps_count, 
    obs_sequence_length=t.obs_sequence_length, 
    action_plan_length=t.action_plan_length,
    actions_count=t.actions_count,
    batches_count=2,
    is_shuffled=False,
    is_next_obs=True,
    is_debug=True,
)

for t.name, t.v in t.dataset.shared_tensors.items():
    if t.v is not None:
        assert t.v.is_shared(), f'{t.name} is not shared'
        LOG(f'{t.name:>28}: {str(t.v.device):>6}, {str(t.v.dtype):>15}, {t.v.shape}')

assert len(t.dataset.valid_inds) == (t.envs_count * t.rollout_steps_count)
assert len(t.dataset.valid_inds.unique()) == (t.envs_count * t.rollout_steps_count)

t.prev_env_valid_inds = None

for t.env_ind in range(t.envs_count):
    t.prev_env_training_ind = lu.when(t.prev_env_valid_inds is not None, lambda: t.prev_env_valid_inds[-1] + 1, 0)
    t.env_valid_inds = t.prev_env_training_ind + (t.obs_sequence_length - 1) + torch.arange(t.rollout_steps_count).to(CONFIG.cuda_device)
    assert torch.all(t.env_valid_inds == t.dataset.valid_inds[t.env_ind*t.rollout_steps_count:(t.env_ind+1)*t.rollout_steps_count])
    t.prev_env_valid_inds = t.env_valid_inds

### Iteration

In [ ]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = 32
t.rollout_steps_count = 128
t.obs_sequence_length = 10
t.action_plan_length = 8
t.actions_count = 6
t.dataset = Dataset(
    envs_count=t.envs_count,
    observation_space_shape=LS.agent.params.observation_space_shape, 
    rollout_steps_count=t.rollout_steps_count, 
    obs_sequence_length=t.obs_sequence_length, 
    action_plan_length=t.action_plan_length,
    actions_count=t.actions_count,
    batches_count=4,
    is_shuffled=True,
    is_next_obs=True,
    is_debug=False,
)

for t.batch in t.dataset:
    pass

for t.fn in t.batch._fields:
    if getattr(t.batch, t.fn) is not None:
        LOG(f'{t.fn:>20}.shape={getattr(t.batch, t.fn).shape}')
    else:
        LOG(f'{t.fn:>20}=None')

# Rollout

## RolloutManager

In [48]:
class RolloutManager:
    def __init__(self, agent, envs_count, rollout_steps_count, prologue_size, random_seed, ram=None, game_modifs=None):
        if ram is not None and game_modifs is not None:
            assert False, 'Only one of "ram" of "game_modifs" can be not None'
        
        self.agent = agent
        self.device = next(iter(agent.parameters())).device
        self.env_inds = torch.arange(envs_count)
        self.rollout_steps_count = rollout_steps_count
        self.prologue_size = prologue_size

        # Sliding windows
        obs_sequence_length = agent.params.obs_sequence_length
        self.w_obs = torch.zeros((envs_count, obs_sequence_length, *self.agent.params.observation_space_shape)).to(self.device)
        self.w_obs_pmasks = torch.ones((envs_count, obs_sequence_length)).to(self.device)
        self.w_actions_noises = self.agent.generate_actions_noise((envs_count, self.agent.params.action_plan_length)).to(self.device)
        
        # Rollout buffers (used only during current rollout, no data forwarding)
        self.r_rewards = torch.zeros((envs_count, self.rollout_steps_count))
        self.r_dones = torch.zeros((envs_count, self.rollout_steps_count))

        thread_pool_size = lu.coalesce_fn(os.environ.get('ALE_THREAD_POOL_SIZE'), int, None)
        thread_affinity_offset = lu.coalesce_fn(os.environ.get('ALE_THREAD_AFFINITY_OFFSET'), int, None)
        self.envs = create_envs(envs_count, thread_pool_size=thread_pool_size, thread_affinity_offset=thread_affinity_offset)
        LOG(f'Envs created: {envs_count} envs, {thread_pool_size=}, {thread_affinity_offset=}')

        if ram is not None:
            reset_options = self.get_reset_rams(self.env_inds, ram)
        elif game_modifs is not None:
            reset_options = self.get_reset_modifs(self.env_inds, game_modifs)
        else:
            reset_options = {}
            
        self.obs, _ = self.envs.reset(seed=random_seed, options=dict(**reset_options))
        self.obs = torch.tensor(self.obs, dtype=torch.float, device=self.device)
        self.obs = Agent.preprocess_obs(self.obs)
        self.w_obs[:,-1] = self.obs
        self.w_obs_pmasks[:,-1] = 0
        LOG('Envs reset')
        
    def rollout(self, shared_tensors, tau, ram=None, game_modifs=None, with_timing_counters=False, with_control_data=False):
        if ram is not None and game_modifs is not None:
            assert False, 'Only one of "ram" of "game_modifs" can be not None'
            
        out_obs, out_obs_pmasks, out_actions, out_action_log_probs, out_values, out_next_action_plan_logits = (
            shared_tensors['obs'],
            shared_tensors['obs_pmasks'],
            shared_tensors['actions'],
            shared_tensors['action_log_probs'],
            shared_tensors['values'],
            shared_tensors['next_action_plan_logits'],
        )
        out_next_obs, out_dones, out_advantages, out_returns = (
            shared_tensors['next_obs'],
            shared_tensors['dones'],
            shared_tensors['advantages'],
            shared_tensors['returns'],
        )
        # debug tensors
        out_debug_w_obs, out_debug_action_plan_logits, out_debug_lookahead_action_plan_logits = (
            shared_tensors['debug_w_obs'],
            shared_tensors['debug_action_plan_logits'],
            shared_tensors['debug_lookahead_action_plan_logits'],
        )
        assert self.rollout_steps_count + self.prologue_size == out_obs_pmasks.shape[1]
        
        # ROLLOUT
        assert torch.all(self.w_obs[:,-1] == self.obs)
        
        # fill prologues with data from the prev rollout
        # For obs and obs_pmasks this is always mandatory. For others - only in case of parallel teacher forcing.
        out_obs[:,:self.prologue_size] = out_obs[:,-self.prologue_size:]
        out_obs_pmasks[:,:self.prologue_size] = out_obs_pmasks[:,-self.prologue_size:]
        out_actions[:,:self.prologue_size] = out_actions[:,-self.prologue_size:]
        out_action_log_probs[:,:self.prologue_size] = out_action_log_probs[:,-self.prologue_size:]
        out_values[:,:self.prologue_size] = out_values[:,-self.prologue_size:]
        out_next_action_plan_logits[:,:self.prologue_size] = out_next_action_plan_logits[:,-self.prologue_size:]
        out_dones[:,:self.prologue_size] = out_dones[:,-self.prologue_size:]
        out_advantages[:,:self.prologue_size] = out_advantages[:,-self.prologue_size:]
        out_returns[:,:self.prologue_size] = out_returns[:,-self.prologue_size:]

        if out_next_obs is not None:
            out_next_obs[:,:self.prologue_size] = out_next_obs[:,-self.prologue_size:]

        life_stats = []
        episode_stats = []
        timing_counters = defaultdict(list)
        control_data = defaultdict(list)

        def collect_stats(key, infos, stats):
            prefix = lu.when('life' in key, 'life', 'episode')
            
            assert len(infos[key]) == len(self.env_inds)
            
            for env_ind in np.argwhere(infos[key]).ravel():
                stats.append(dict(
                    env_ind=env_ind.item(),
                    r=infos[prefix + '_returns'][env_ind],
                    l=infos[prefix + '_lengths'][env_ind],
                ))

        with torch.no_grad():
            for rollout_step in range(self.rollout_steps_count):
                t0 = time.time()
                
                out_index = self.prologue_size + rollout_step
                out_obs[:,out_index] = self.obs
                out_obs_pmasks[:,out_index] = self.w_obs_pmasks
                
                if out_debug_w_obs is not None:
                    out_debug_w_obs[:,out_index] = self.w_obs

                timing_counters['out_obs'].append(time.time() - t0)
                t0 = time.time()

                # Analyze observation windows
                agent_result = self.agent(
                    obs=self.w_obs, 
                    padding_masks=self.w_obs_pmasks,
                    actions_noise=self.w_actions_noises[:,0],
                    tau=tau,
                )

                out_actions[:,out_index] = agent_result.actions
                out_action_log_probs[:,out_index] = agent_result.action_log_probs
                out_values[:,out_index] = agent_result.values

                if out_debug_action_plan_logits is not None:
                    out_debug_action_plan_logits[:,out_index] = agent_result.action_plan_logits

                if rollout_step > 0: # population of out_next_action_plan_logits is also done during ADVANTAGES calc
                    assert (out_index - 1) >= 0
                    # Here we must patch PREVIOUS record!
                    # Note: if we just started from resetted env then we will patch record which belong to old env which is not quite correct.
                    # We workaround this during optimization by looking at dones flag - we won't take into account next_action_plan_logits for terminal records
                    out_next_action_plan_logits[:,out_index-1] = agent_result.action_plan_logits  

                if with_control_data:
                    control_data['actions'].extend(agent_result.actions.ravel().tolist())
                    control_data['action_log_probs'].extend(agent_result.action_log_probs.ravel().tolist())
                    control_data['values'].extend(agent_result.values.ravel().tolist())
                
                timing_counters['get_action_and_value'].append(time.time() - t0)
                t0 = time.time()
                
                # Interact with environments
                self.obs, rewards, terminations, truncations, infos = self.envs.step(agent_result.actions.cpu().numpy().ravel())
                self.obs = torch.tensor(self.obs, dtype=torch.float, device=self.device)
                self.obs = Agent.preprocess_obs(self.obs)
                
                # Update sliding windows which are used to feed agent
                self.w_obs = self.w_obs.roll(shifts=-1, dims=1)
                self.w_obs[:,-1] = self.obs
                
                self.w_obs_pmasks = self.w_obs_pmasks.roll(shifts=-1, dims=1)
                self.w_obs_pmasks[:,-1] = 0

                self.w_actions_noises = self.w_actions_noises.roll(shifts=-1, dims=1)
                self.w_actions_noises[:,-1] = self.agent.generate_actions_noise(len(self.env_inds))

                if out_next_obs is not None:
                    out_next_obs[:,out_index] = self.obs
                    
                need_reset_envs = np.logical_or(terminations, truncations)
                dones = need_reset_envs.copy()

                if HP.env.is_episodic_life and np.any(infos['is_life_lost']):
                    # Manually trigger done flag and truncate observations window (i.e. leave only current observation for new life)
                    # for envs which hit life loss
                    life_lost_envs = infos['is_life_lost']
                    self.w_obs_pmasks[life_lost_envs] = 1
                    self.w_obs_pmasks[life_lost_envs,-1] = 0
                    dones[life_lost_envs] = True
                
                self.r_dones[:,rollout_step] = torch.tensor(dones, dtype=torch.float)
                self.r_rewards[:,rollout_step] = torch.tensor(rewards, dtype=torch.float)

                if with_control_data:
                    control_data['dones'].extend(torch.tensor(dones, dtype=torch.float).tolist())
                
                collect_stats('is_life_lost', infos, life_stats)
                collect_stats('is_episode_over', infos, episode_stats)
                timing_counters['env.step'].append(time.time() - t0)
        
                # Reset envs which hit an episode end or are truncated manually, for resetted envs get new observations
                if np.any(need_reset_envs):
                    t00 = time.time()

                    if ram is not None:
                        reset_options = self.get_reset_rams(np.argwhere(need_reset_envs).ravel(), ram)
                    elif game_modifs is not None:
                        reset_options = self.get_reset_modifs(np.argwhere(need_reset_envs).ravel(), game_modifs)
                    else:
                        reset_options = {}

                    reset_obs, _ = self.envs.reset(options=dict(reset_mask=need_reset_envs, **reset_options))
                    reset_obs = torch.tensor(reset_obs[need_reset_envs], dtype=torch.float).to(self.device)
                    reset_obs = Agent.preprocess_obs(reset_obs)
                    self.obs[need_reset_envs] = reset_obs
                    self.w_obs[:,-1] = self.obs
                    self.w_obs_pmasks[need_reset_envs] = 1
                    self.w_obs_pmasks[need_reset_envs,-1] = 0
                    timing_counters['env.reset'].append(time.time() - t00)

        t0 = time.time()
        
        # ADVANTAGES
        with torch.no_grad():
            # Look ahead for one step
            agent_result = self.agent(
                obs=self.w_obs,
                padding_masks=self.w_obs_pmasks,
                actions_noise=self.agent.generate_actions_noise(len(self.env_inds)), # noise actually doesn't matter here because we need action_plan_logits only
                tau=tau,
            )
            out_next_action_plan_logits[:,-1] = agent_result.action_plan_logits 

            if out_debug_lookahead_action_plan_logits is not None:
                out_debug_lookahead_action_plan_logits[:,0] = agent_result.action_plan_logits
            
            rewards = self.r_rewards.to(self.device, non_blocking=True)
            dones = self.r_dones.to(self.device, non_blocking=True)
            values = out_values[:,-self.rollout_steps_count:]
            advantages = torch.zeros_like(out_advantages[:,-self.rollout_steps_count:])
            assert rewards.shape[1] == self.rollout_steps_count
            assert dones.shape[1] == self.rollout_steps_count
            assert values.shape[1] == self.rollout_steps_count
            assert advantages.shape[1] == self.rollout_steps_count
            lastgaelam = 0
            
            for t in reversed(range(self.rollout_steps_count)):
                if t == self.rollout_steps_count - 1:
                    nextnonterminal = 1.0 - dones[:,t]
                    nextnonterminal = nextnonterminal.to(self.device, non_blocking=True)
                    nextvalues = agent_result.values
                else:
                    nextnonterminal = 1.0 - dones[:,t]
                    nextvalues = values[:,t+1]
                
                delta = rewards[:,t] + HP.ppo.gamma * nextvalues * nextnonterminal - values[:,t]
                lastgaelam = delta + HP.ppo.gamma * HP.ppo.gae_lambda * nextnonterminal * lastgaelam
                advantages[:,t] = lastgaelam

            out_advantages[:,-self.rollout_steps_count:] = advantages
            returns = advantages + out_values[:,-self.rollout_steps_count:]
            out_returns[:,-self.rollout_steps_count:] = returns
            out_dones[:,-self.rollout_steps_count:] = dones

        if with_control_data:
            control_data['advantages'].extend(advantages.ravel().tolist())
            control_data['returns'].extend(returns.ravel().tolist())
            
        timing_counters['advantages'].append(time.time() - t0)

        return dict(
            life_stats=life_stats,
            episode_stats=episode_stats,
            timing_counters=lu.when(with_timing_counters, timing_counters, None),
            control_data=lu.when(with_control_data, control_data, None),
        )

    @staticmethod
    def get_reset_rams(env_inds, ram):
        if ram is None:
            return {}

        return dict(reset_rams=dict(map(lambda env_ind: (int(env_ind), ram), env_inds)))

    @staticmethod
    def get_reset_modifs(env_inds, modifs):
        if modifs is None:
            return {}

        return dict(reset_modifs=dict(map(lambda env_ind: (int(env_ind), modifs), env_inds)))

## Test

### rollout: performance

In [ ]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = lu.when(CONFIG.is_cuda, 32, 8)
t.timing_counters = defaultdict(list)
t.rollout_steps_count = 128
t.dataset = Dataset(
    envs_count=t.envs_count,
    observation_space_shape=LS.agent.params.observation_space_shape, 
    rollout_steps_count=t.rollout_steps_count, 
    obs_sequence_length=LS.agent.params.obs_sequence_length, 
    action_plan_length=LS.agent.params.action_plan_length,
    actions_count=LS.agent.params.actions_count,
    batches_count=4,
    is_shuffled=True,
    is_next_obs=True,
    is_debug=False,
)
t.rm = RolloutManager(
    agent=LS.agent, 
    envs_count=t.envs_count, 
    rollout_steps_count=t.rollout_steps_count, 
    prologue_size=t.dataset.prologue_size,
    random_seed=HP.system.random_seed,
)

t.global_steps_count = t.rollout_steps_count * t.envs_count * 10
t.step = 0

with tqdm(total=t.global_steps_count) as pbar:
    while t.step < t.global_steps_count:
        t.rr = t.rm.rollout(t.dataset.shared_tensors, tau=0.1, with_timing_counters=True)

        for t.timing_counter_key, t.timing_counter_times in t.rr['timing_counters'].items():
            t.timing_counters[t.timing_counter_key].extend(t.timing_counter_times)

        step_inc = t.rollout_steps_count * t.envs_count
        t.step += step_inc
        pbar.update(step_inc)

list(map(lambda kv: (kv[0], np.array(kv[1]).mean().item()), t.timing_counters.items()))

### rollout: obs, pmasks, actions, etc.

In [ ]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = lu.when(CONFIG.is_cuda, 32, 8)
t.df_columns = defaultdict(list)
t.all_life_stats_items = []
t.all_episode_stats_items = []
t.rollout_steps_count = 128
t.dataset = Dataset(
    envs_count=t.envs_count,
    observation_space_shape=LS.agent.params.observation_space_shape, 
    rollout_steps_count=t.rollout_steps_count, 
    obs_sequence_length=LS.agent.params.obs_sequence_length, 
    action_plan_length=LS.agent.params.action_plan_length,
    actions_count=LS.agent.params.actions_count,
    batches_count=4,
    is_shuffled=False,
    is_next_obs=True,
    is_debug=True,
)
t.rm = RolloutManager(
    agent=LS.agent, 
    envs_count=t.envs_count, 
    rollout_steps_count=t.rollout_steps_count, 
    prologue_size=t.dataset.prologue_size,
    random_seed=HP.system.random_seed,
)

t.global_steps_count = t.rollout_steps_count * t.envs_count * 5
t.step = 0

with tqdm(total=t.global_steps_count) as pbar:
    while t.step < t.global_steps_count:
        t.stats_afs = dict(
            life_stats=dict(l=RecursiveAverageFilter(), r=RecursiveAverageFilter()),
            episode_stats=dict(l=RecursiveAverageFilter(), r=RecursiveAverageFilter()),
        )

        t.rr = t.rm.rollout(t.dataset.shared_tensors, tau=0.1, with_control_data=True)

        for t.k in t.stats_afs:
            for t.stats_item in t.rr[t.k]:
                t.stats_afs[t.k]['l'](t.stats_item['l'])
                t.stats_afs[t.k]['r'](t.stats_item['r'])

                if t.k == 'episode_stats':
                    t.all_episode_stats_items.append(t.stats_item['r'])

                if t.k == 'life_stats':
                    t.all_life_stats_items.append(t.stats_item['r'])

        t.obs_pmasks = t.dataset.obs_pmasks.view(-1, *t.dataset.obs_pmasks.shape[2:])
        t.debug_w_obs = t.dataset.debug_w_obs.view(-1, *t.dataset.debug_w_obs.shape[2:])

        # Verify that we get exactly the same windows (obs, obs_pmasks) as during rollout
        for t.batch in t.dataset:
            assert torch.all(t.obs_pmasks[t.batch.b_inds] == t.batch.obs_pmasks)
            assert torch.all(t.batch.obs_pmasks[:,-1] == 0) # current items are last items and are always unmasked, make sure obs_pmasks reflects this fact
            
            t.b_orig_w_obs = t.debug_w_obs[t.batch.b_inds]
            assert torch.all(t.b_orig_w_obs == t.batch.obs)

        # Verify other data via control data
        actions = []
        action_log_probs = []
        values = []
        dones = []
        advantages = []
        returns = []
        
        for t.batch in t.dataset:
            actions.extend(t.batch.actions.cpu().numpy())
            action_log_probs.extend(t.batch.action_log_probs.cpu().numpy())
            values.extend(t.batch.values.cpu().numpy())
            dones.extend(t.batch.dones.cpu().numpy())
            advantages.extend(t.batch.advantages.cpu().numpy())
            returns.extend(t.batch.returns.cpu().numpy())

        assert np.all(np.sort(np.array(actions)) == np.sort(t.rr['control_data']['actions']))
        assert np.all(np.sort(np.array(action_log_probs)) == np.sort(t.rr['control_data']['action_log_probs']))
        assert np.all(np.sort(np.array(values)) == np.sort(t.rr['control_data']['values']))
        assert np.all(np.sort(np.array(dones)) == np.sort(t.rr['control_data']['dones']))
        assert np.all(np.sort(np.array(advantages)) == np.sort(t.rr['control_data']['advantages']))
        assert np.all(np.sort(np.array(returns)) == np.sort(t.rr['control_data']['returns']))
            
        step_inc = t.rollout_steps_count * t.envs_count
        t.step += step_inc
        pbar.update(step_inc)

        t.df_columns['life_stats.r'].append(t.stats_afs['life_stats']['r'].v)
        t.df_columns['life_stats.l'].append(t.stats_afs['life_stats']['l'].v)
        t.df_columns['life_stats.n'].append(t.stats_afs['life_stats']['l'].n)
        t.df_columns['episode_stats.r'].append(t.stats_afs['episode_stats']['r'].v)
        t.df_columns['episode_stats.l'].append(t.stats_afs['episode_stats']['l'].v)
        t.df_columns['episode_stats.n'].append(t.stats_afs['episode_stats']['l'].n)

pd.DataFrame(t.df_columns).style.format("{:.2f}")

### rollout: next_obs, next_action_plan_logits

In [ ]:
# @launchit.disable
t = lu.ScopedVars()
t.rollout_steps_count = 256
t.dataset = Dataset(
    envs_count=1,
    observation_space_shape=LS.agent.params.observation_space_shape, 
    rollout_steps_count=t.rollout_steps_count, 
    obs_sequence_length=LS.agent.params.obs_sequence_length, 
    action_plan_length=LS.agent.params.action_plan_length,
    actions_count=LS.agent.params.actions_count,
    batches_count=1,
    is_shuffled=False,
    is_next_obs=True,
    is_debug=True,
)
t.rm = RolloutManager(
    agent=LS.agent, 
    envs_count=1, 
    rollout_steps_count=t.rollout_steps_count, 
    prologue_size=t.dataset.prologue_size,
    random_seed=HP.system.random_seed,
)
    
t.global_steps_count = t.rollout_steps_count * 4
t.step = 0
t.any_dones = False

with tqdm(total=t.global_steps_count) as pbar:
    while t.step < t.global_steps_count:
        t.rr = t.rm.rollout(t.dataset.shared_tensors, tau=0.1)

        t.batch = next(iter(t.dataset)) # one batch only
        
        # Internal check: verify that within window items which are not terminal and which are present (obs_pmasks) have matching next obs
        # Disabled because one need to return is_ptf for Dataset
        # t.bw_inds = t.dataset.bw_inds_stem + t.batch.b_inds.unsqueeze(1)
        # t.bw_inds = t.bw_inds.view(-1)
        # t.bw_dones = t.dones[t.bw_inds].view(t.batch.obs.shape[:2])

        # for t.i in range(t.bw_dones.shape[0]):
        #     for t.j in range(t.bw_dones.shape[1] - 1):
        #         if t.bw_dones[t.i,t.j] == 0 and t.batch.obs_pmasks[t.i,t.j] == 0:
        #             assert torch.all(t.batch.obs[t.i,t.j+1] == t.batch.next_obs[t.i,t.j])
        
        # External check: verify that all running (non terminal) items must have next item where obs == next_obs
        t.any_dones = t.any_dones or torch.any(t.batch.dones.bool())
        t.running = ~t.batch.dones.bool()
        assert len(t.running) == len(t.batch.obs)
        
        for i, is_running in enumerate(t.running):
            if is_running:
                if (i + 1) < len(t.batch.obs):
                    assert torch.all(t.batch.next_obs[i,-1] == t.batch.obs[i+1,-1])
                else:
                    # Don't have access to next obs since this is a last item in rollout batch
                    pass
            else:
                # We've done env, t.batch.next_obs could of any data - it will be ignored anyway
                pass 
            

        t.debug_action_plan_logits = t.dataset.debug_action_plan_logits.view(-1, *t.dataset.debug_action_plan_logits.shape[2:])
        t.debug_action_plan_logits = t.debug_action_plan_logits[t.batch.b_inds]
        assert len(t.running) == len(t.batch.next_action_plan_logits)
        
        for i, is_running in enumerate(t.running):
            if is_running:
                if (i + 1) < len(t.batch.next_action_plan_logits):
                    assert torch.all(t.batch.next_action_plan_logits[i] == t.debug_action_plan_logits[i+1])
                else:
                    assert torch.all(t.batch.next_action_plan_logits[i] == t.dataset.debug_lookahead_action_plan_logits)
            else:
                # We've done env, t.batch.next_action_plan_logits could of any data - it will be ignored anyway
                pass 
            
        step_inc = t.rollout_steps_count
        t.step += step_inc
        pbar.update(step_inc)

assert t.any_dones

# Video (via MP)

Implementation details regarding memory sharing between PyTorch applications: <a href="./dialogs/torch-multiprocessing-shm.ipynb">torch-multiprocessing-shm.ipynb</a>

## generate_video_file_name

In [23]:
def generate_video_file_name():
    lc = HP.launch_component()
    timestamp = datetime.datetime.now().strftime("%Y.%m.%d-%H:%M:%S")
    return os.path.join(CONFIG.run_path, f'video-{lc.name}-launch{lc.version}-{timestamp}.mp4')

## capture_video_of_test_rollout

In [24]:
def capture_video_of_test_rollout(agent, max_steps_count, tau, 
                                  video_file_name=None, random_seed=None, fps=30, rams=None, capture_preprocessed_obs=False, break_on_level_passed=False):
    device = next(iter(agent.parameters())).device
    video_file_name = lu.coalesce(video_file_name, lambda: generate_video_file_name())
    assert video_file_name is not None
    
    captured_frames = []
    game_meta = defaultdict(int)
    
    env = create_envs(1, track_levels=True)
    env.reset(seed=random_seed)
    try:
        rams = lu.when(rams is None or len(rams) == 0, [None], rams)
        
        for ram_ind, ram in enumerate(rams):
            reset_rams = lu.when(ram is not None, lambda: {0: ram}, {})
            LOG(f'Capturing video for RAM #{ram_ind}: {ram}')
            obs, _ = env.reset(options=dict(reset_rams=reset_rams))
            obs = torch.tensor(obs).to(device)
            obs = Agent.preprocess_obs(obs.squeeze(0))
            obs_buf = torch.zeros((agent.params.obs_sequence_length, *agent.params.observation_space_shape)).to(device)
            obs_buf[-1] = obs
            obs_pmask = torch.ones(agent.params.obs_sequence_length).to(device)
            obs_pmask[-1] = 0
            levels_passed = 0
            
            with torch.no_grad():
                for step in range(max_steps_count): 
                    if step % 100 == 0:
                        LOG(f'Captured 100 steps, total steps captured {step}')
                        
                    agent_result = agent(
                        obs=obs_buf.unsqueeze(0),
                        padding_masks=obs_pmask.unsqueeze(0),
                        actions_noise=agent.generate_actions_noise(1),
                        tau=tau,
                    )
            
                    raw_obs, reward, terminated, truncated, info = env.step(agent_result.actions.cpu().numpy())
                    obs = torch.tensor(raw_obs).to(device)
                    obs = Agent.preprocess_obs(obs.squeeze(0))

                    if capture_preprocessed_obs:
                        assert obs[0].ndim == 2 # e.g. (84, 84)
                        captured_frames.append(obs[0])
                    else:
                        captured_frames.append(raw_obs[0])
                    
                    obs_buf = obs_buf.roll(shifts=-1, dims=0)
                    obs_buf[-1] = obs
                    
                    obs_pmask = obs_pmask.roll(shifts=-1, dims=0)
                    obs_pmask[-1] = 0
            
                    if HP.env.is_episodic_life and info['is_life_lost'][0]:
                        obs_pmask.fill_(1)
                        obs_pmask[-1] = 0

                    if info['level_passed'][0]:
                        levels_passed += 1

                        if break_on_level_passed:
                            break

                    if terminated or truncated:
                        assert info['is_episode_over'][0], info
                        break
            
            game_meta['reward'] += info['episode_returns'][0].item()
            game_meta['frames_count'] += info['episode_lengths'][0].item()
            game_meta['steps_count'] += step + 1
            game_meta['levels_passed'] += levels_passed

        if capture_preprocessed_obs:
            captured_frames = torch.stack(captured_frames, axis=0)
            captured_frames *= 255
            captured_frames = captured_frames.to(torch.uint8)
            captured_frames = torch.stack([captured_frames] * 3, axis=-1) # duplicate last channel (N, 84, 84) -> (N, 84, 84, 3)
            captured_frames = list(captured_frames.cpu().numpy())
        
        clip = ImageSequenceClip(captured_frames, fps=fps / env.frame_skip) 
        try:
            clip.write_videofile(video_file_name, logger=None)
        finally:
            clip.close()  # Explicitly closes ffmpeg processes and frees memory
                
        video_meta = {}
        
        with open(video_file_name, 'rb') as f:
            container = av.open(f)
            video_meta['fps'] = float(container.streams.video[0].average_rate)
            video_stream = container.streams.video[0]
            video_meta['duration'] = float(video_stream.duration * video_stream.time_base)
            
        return video_file_name, dict(game=game_meta, video=video_meta)
    finally:
        del env # nudge Python to release threads and other resources created by EnvVectorizer inside AtariVectorEnv

## WorkerTask

In [25]:
# Exchange data between main and child processes
@dataclass(slots=True)
class WorkerTask:
    task_id: int
    op: str
    params: dict = None

@dataclass(slots=True)
class WorkerTaskResult:
    task_id: int
    payload: object = None

## WorkerCtl

In [26]:
# Master's stuff (main process)
class WorkerCtl:
    task_id = 0
    
    def __init__(self, worker_ind, module, mp_ctx, enabled_loggers):
        self.task_ctor = getattr(module, 'WorkerTask')
        self.worker_ind = worker_ind
        self.task_queue = mp_ctx.Queue()
        self.task_result_queue = mp_ctx.Queue()
        self.process = mp_ctx.Process(
            target=getattr(module, 'worker_loop'), 
            args=(worker_ind, self.task_queue, self.task_result_queue, enabled_loggers),
        )
        self.process.start()
        self.pending_task_ids = deque()

    @staticmethod
    def gen_task_id():
        WorkerCtl.task_id += 1
        return WorkerCtl.task_id

    def healthcheck(self):
        task = self.task_ctor(task_id=self.gen_task_id(), op='HEALTHCHECK')
        self.task_queue.put(task)
        self.task_result_queue.get()
        
    def terminate(self, timeout=None):
        if self.process.is_alive():
            task = self.task_ctor(task_id=self.gen_task_id(), op='TERMINATE')
            try:
                self.task_queue.put(task)
                self.task_result_queue.get(timeout=timeout)
                self.process.join()
            except:
                self.process.terminate()

    def init_agent(self, agent_params, agent_state_dict=None, torch_compile=False, device=None):
        if agent_state_dict is not None:
            for key in agent_state_dict:
                assert agent_state_dict[key].is_shared()
        
        task = self.task_ctor(
            task_id=self.gen_task_id(), 
            op='INIT_AGENT', 
            params=dict(
                agent_params=dataclasses.asdict(agent_params),
                agent_state_dict=agent_state_dict,
                torch_compile=torch_compile,
                device=device,
            ),
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id

    def sync_agent(self, agent_state_dict):
        for key in agent_state_dict:
            assert agent_state_dict[key].is_shared()
                
        task = self.task_ctor(
            task_id=self.gen_task_id(),
            op='SYNC_AGENT',
            params=dict(agent_state_dict=agent_state_dict),
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id

    def capture_video_of_test_rollout(self, max_steps_count, tau, video_file_name, 
                                      random_seed=None, rams=None, forward_data=None, capture_preprocessed_obs=False, break_on_level_passed=False):
        task = self.task_ctor(
            task_id=self.gen_task_id(),
            op='CAPTURE_VIDEO',
            params=dict(
                max_steps_count=max_steps_count, 
                tau=tau,
                video_file_name=video_file_name,
                random_seed=random_seed,
                rams=rams,
                forward_data=forward_data,
                capture_preprocessed_obs=capture_preprocessed_obs,
                break_on_level_passed=break_on_level_passed,
            ),
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id

    def is_busy(self):
        return len(self.pending_task_ids) > 0
    
    def get_task_result(self):
        assert len(self.pending_task_ids) > 0
        result = self.task_result_queue.get()
        assert result.task_id == self.pending_task_ids.popleft()
        return result

    def peek_task_result(self):
        if not self.pending_task_ids:
            return None
        
        try:
            result = self.task_result_queue.get(block=False)
            assert result.task_id == self.pending_task_ids.popleft()
            return result
        except queue.Empty:
            return None # task is not completed yet

    def drain_task_results(self):
        results = []

        while self.pending_task_ids:
            task_id = self.pending_task_ids.popleft()
            task_result = self.task_result_queue.get()
            assert task_result.task_id == task_id
            results.append(task_result)

        return results

## worker_loop

In [27]:
# Executed in child process
def worker_loop(worker_ind, task_queue, task_result_queue, enabled_loggers=()):
    @dataclass(slots=True)
    class WorkerState:
        agent: object = None
        is_attached_agent: bool = False
    
    CONFIG = create_config()
    global LOG # push to global so it's available to RolloutManager
    LOG = Logging.get()
    LOG.app_name = CONFIG.self_name

    for x in ('syslog', 'stdout', 'verbose_stdout'):
        LOG.enable(x, x in enabled_loggers)
        
    worker_random_seed = HP.system.random_seed + worker_ind

    with LOG.auto_prefix('WRK', worker_ind, 'SEED', worker_random_seed):
        LOG(f'CONFIG={CONFIG._asdict()}')
        
        au.init()
        random.seed(worker_random_seed)
        torch.manual_seed(worker_random_seed)
        RNG = np.random.default_rng(worker_random_seed)
        LOG(f'{worker_random_seed=}')
        
        torch.backends.cudnn.deterministic = HP.system.is_torch_deterministic
        LOG(f'{torch.backends.cudnn.deterministic=}')

        WS = WorkerState()
        LOG('Worker is ready')
        
        task_wait_timeout = 60
        is_running = True
    
        while is_running:
            try:
                # task is expected to be an instanace of WorkerTask class
                task = task_queue.get(block=True, timeout=task_wait_timeout)
            except queue.Empty:
                LOG(f'Didn\'t get any tasks within {task_wait_timeout} seconds, waiting again')
                continue

            with LOG.auto_prefix('TASK', task.task_id):
                LOG(f'Got task #{task.task_id} {task.op}')
                task_result = WorkerTaskResult(task_id=task.task_id)
                
                match task.op:
                    case 'HEALTHCHECK':
                        pass
                    case 'TERMINATE':
                        is_running = False
                    case 'INIT_AGENT':
                        ap = Agent.Params(**task.params['agent_params'])
                        device = lu.coalesce(task.params.get('device'), CONFIG.cuda_device)
                        WS.agent = Agent(ap).to(device)
                        
                        if task.params['torch_compile']:
                            WS.agent = torch.compile(WS.agent, fullgraph=True)
                            LOG('Agent compiled')
                            
                        WS.is_attached_agent = task.params['agent_state_dict'] is not None
                        
                        if WS.is_attached_agent:
                            # Storage for weights of attached agent is pointed to agent's weights in main process
                            state_dict = task.params['agent_state_dict']
                            
                            with torch.no_grad():
                                for name, param in WS.agent.named_parameters():
                                    assert state_dict[name].is_shared()
                                    param.data = state_dict[name]

                        LOG(f'{WS.is_attached_agent=}')
                    case 'SYNC_AGENT':
                        assert WS.agent is not None
                        assert not WS.is_attached_agent
                        
                        state_dict = task.params['agent_state_dict']
                        
                        with torch.no_grad():
                            # Fast GPU-TO-GPU sync
                            for name, param in WS.agent.named_parameters():
                                assert state_dict[name].is_shared()
                                param.copy_(state_dict[name])
                    case 'CAPTURE_VIDEO':
                        assert WS.agent is not None
                        LOG(f'Starting video capture: {task.params=}')
                        with torch.no_grad():
                            video_file_name, video_meta = capture_video_of_test_rollout(
                                WS.agent, 
                                max_steps_count=task.params['max_steps_count'], 
                                tau=task.params['tau'],
                                video_file_name=task.params['video_file_name'],
                                random_seed=task.params['random_seed'],
                                rams=task.params['rams'],
                                capture_preprocessed_obs=task.params['capture_preprocessed_obs'],
                                break_on_level_passed=task.params['break_on_level_passed'],
                            )
                            task_result.payload = (video_file_name, video_meta, task.params['forward_data'])
                    case _:
                        LOG(f'Unknown {task.op=}, ignoring')
        
                task_result_queue.put(task_result)
                LOG('Task complete')
        
        LOG('Worker is going down')

## get_worker_factory

In [29]:
def get_worker_factory():
    with LOG.auto_log_level(logging.INFO):
        expandvars = dict(
            PROJECT_ROOT_PATH=CONFIG.project_root_path,
            MODEL_NAME=CONFIG.self_name,
            MODEL_VERSION=HP.launch_component().version,
            LAUNCH_GOAL=LaunchGoal.WORKER.value,
        )
        module_fname = launchit.launchit(
            CONFIG.self_fname, 
            expandvars=expandvars, 
            make_py_file=True,
            dir_name=CONFIG.run_path,
            collect_inds=[],
            disable_inds=['worker']
        )
        LOG.info(f'Created "{module_fname}"')
        
    module_dir_name = os.path.dirname(module_fname)
    module_name = os.path.splitext(os.path.basename(module_fname))[0]
    sys.path.append(module_dir_name)
    module = __import__(module_name)

    def factory(worker_ind):
        if CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK:
            enabled_loggers = ('verbose_stdout', )
        else:
            enabled_loggers = ('syslog', )
        return WorkerCtl(worker_ind, module, LS.mp_ctx, enabled_loggers)

    return factory

## Test

### capture_video_of_test_rollout

In [32]:
# @launchit.disable
level1_ram = np.array(
    [  0,  96, 158,   0,  15, 144, 255, 152, 255, 160, 255, 168, 255,
       176, 255, 184, 255,  16,  80,  16,  80,  16,   0,   0,   0,   2,
       3,   4,   0, 192,   8,  16,  80,  16,  80,   0,   0,   0,   0,
       0,   0,   0,   0,  12,  12,  12,  12, 140,   7,   7,   7,   7,
       7,   7,   7,   7, 255, 255,   0, 132,  10,  10,  10,   0,   0,
       0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   3, 255,
       0,   0,   0,   0,   0, 255,   0,   0,   0,   0,   0,   0,   0,
       0,   1,   1,   1,   1, 160, 192, 176, 208,  27,  69,  64,  38,
       140,   0,   0,   0,   0,   0,   0,   0,   0,   0, 255,   0,   0,
       0,   0,   0,   4, 255, 255,   0, 212, 255, 254, 245], dtype=np.uint8)
capture_video_of_test_rollout(
    LS.agent, 
    max_steps_count=300, 
    tau=0.1, 
    rams=[None, level1_ram], 
    capture_preprocessed_obs=True,
)

Capturing video for RAM #0: None
Captured 100 steps, total steps captured 0
Captured 100 steps, total steps captured 100
Captured 100 steps, total steps captured 200
Capturing video for RAM #1: [  0  96 158   0  15 144 255 152 255 160 255 168 255 176 255 184 255  16
  80  16  80  16   0   0   0   2   3   4   0 192   8  16  80  16  80   0
   0   0   0   0   0   0   0  12  12  12  12 140   7   7   7   7   7   7
   7   7 255 255   0 132  10  10  10   0   0   0   0   0   0   0   0   0
   0   0   0   0   3 255   0   0   0   0   0 255   0   0   0   0   0   0
   0   0   1   1   1   1 160 192 176 208  27  69  64  38 140   0   0   0
   0   0   0   0   0   0 255   0   0   0   0   0   4 255 255   0 212 255
 254 245]
Captured 100 steps, total steps captured 0
Captured 100 steps, total steps captured 100
Captured 100 steps, total steps captured 200
CPU times: user 22.9 s, sys: 23.1 ms, total: 22.9 s
Wall time: 2.48 s


('/home/misha/dev/mine/neurolab/run/17_rl/video-17e_ppo_tr_atari_mp_13-launch0-2026.07.03-16:40:08.mp4',
 {'game': defaultdict(int,
              {'reward': 10.0, 'frames_count': 2451, 'steps_count': 600}),
  'video': {'fps': 7.5, 'duration': 80.0}})

### async capture_video_of_test_rollout

In [161]:
# @launchit.disable
wf = get_worker_factory()
t.worker = wf(0)

try:
    t.worker.init_agent(LS.agent.params)
    t.worker.get_task_result()
    
    video_file_name = generate_video_file_name()
    # _orig_mod may be introduced when using torch.compile()
    t.state_dict = lu.when(hasattr(LS.agent, '_orig_mod'), lambda: LS.agent._orig_mod.state_dict(), lambda: LS.agent.state_dict())
    t.tid = t.worker.sync_agent(t.state_dict)
    t.worker.capture_video_of_test_rollout(max_steps_count=3000, tau=0.1, video_file_name=video_file_name, capture_preprocessed_obs=True)
    t.tr = t.worker.get_task_result() # wait for weights are synced is done
    assert t.tr.task_id == t.tid
    t.result = t.worker.drain_task_results()[-1]
    print(t.result)
finally:
    t.worker.terminate()

Created "/home/misha/dev/mine/neurolab/run/17_rl/17e_ppo_tr_atari_mp_12-launch3.py"


/home/misha/anaconda3/envs/mine/lib/python3.12/site-packages/cupyx/scipy/__init__.py:10: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  import scipy as _scipy


WorkerTaskResult(task_id=3, payload=('/home/misha/dev/mine/neurolab/run/17_rl/video-17e_ppo_tr_atari_mp_12-launch0-2026.07.01-22:15:38.mp4', {'game': defaultdict(<class 'int'>, {'reward': 60.0, 'frames_count': 1549, 'steps_count': 381}), 'video': {'fps': 7.5, 'duration': 50.8}}, None))


## CaptureVideoManager

In [ ]:
class CaptureVideoManager:
    def __init__(self, capture_video_policy, capture_env_rams):
        ump = hp_parse_universal_module(capture_video_policy)
        self.should_capture_video = self.create_should_capture_video(ump.module_name, *ump.args, **ump.kwargs)
        
        wf = get_worker_factory()
        self.worker = wf(0)
        LOG(f'Capture video worker created')

        self.worker.init_agent(agent_params=LS.agent.params)
        self.worker.get_task_result()
        LOG(f'Capture video worker initialized')

        self.capture_env_rams = capture_env_rams

    def is_busy(self):
        return self.worker.is_busy()
    
    def schedule_capture_video(self, tau, global_step):
        assert not self.worker.is_busy()
        # _orig_mod may be introduced when using torch.compile()
        state_dict = lu.when(hasattr(LS.agent, '_orig_mod'), lambda: LS.agent._orig_mod.state_dict(), lambda: LS.agent.state_dict())
        task_id = self.worker.sync_agent(state_dict)
        self.worker.capture_video_of_test_rollout(
            max_steps_count=10_000, 
            tau=tau,
            video_file_name=generate_video_file_name(),
            random_seed=HP.system.random_seed + HP.env.count + 1,
            rams=self.capture_env_rams,
            forward_data=dict(global_step=global_step),
            capture_preprocessed_obs=HP.video.capture_preprocessed_obs,
            break_on_level_passed=HP.video.break_on_level_passed,
        )
        task_result = self.worker.get_task_result() # wait until sync weights is finished so we capture video on agent with weights as LS.agent
        assert task_id == task_result.task_id

    def upload_captured_video(self, is_drain=False):
        if is_drain:
            trs = self.worker.drain_task_results()
        else:
            trs = [self.worker.peek_task_result()]

        metrics = dict(reward=[], levels_passed=[])

        for tr in filter(lambda tr: tr is not None, trs):
            video_fname, video_meta, forward_data = tr.payload

            for metric_name in metrics:
                metrics[metric_name].append(video_meta['game'][metric_name])
                
            _, video_fname_ext = os.path.splitext(video_fname)
            ts = datetime.datetime.now().strftime('%Y.%m.%d-%H:%M:%S')
            remote_video_fname = f'{ts}-{forward_data['global_step']:09}.{video_fname_ext.lstrip('.')}'
            LS.summary_writer.add_file(video_fname, remote_video_fname)
            LS.summary_writer.add_file(io.StringIO(json.dumps(video_meta)), remote_video_fname + '.meta')
            ref_text = f'<a href="http://tensorboard-videos:6007/{LS.summary_writer.log_dir}/{remote_video_fname}" target="_blank">{remote_video_fname}</a>'
            LS.summary_writer.add_text('videos', ref_text, forward_data['global_step'])

        return metrics

    @staticmethod
    def create_should_capture_video(policy_name, period=None):
        if policy_name == 'every':
            assert period is not None
            last_count = 0
            
            def every_thunk(count, is_last_step):
                nonlocal last_count
                elapsed = count - last_count

                if count == 0 or elapsed >= period:
                    last_count = count
                    return True

                return is_last_step

            return every_thunk
        elif policy_name == 'never':
            def never_thunk(count, is_last_step):
                return False

            return never_thunk
        
        assert False, f'Unsupported {policy_name=}'

## Configure

In [ ]:
# @launchit.disable
# @launchit.collect

# Video params
HP.video.capture_policy = 'every(1000000)' # video capture policy depending on steps
HP.video.capture_preprocessed_obs = False
HP.video.capture_env_rams = [
    'com.develorium.neurolab.frostbite_ram:level1_101:1',
    'com.develorium.neurolab.frostbite_ram:level4_101:1',
    'com.develorium.neurolab.frostbite_ram:level5_101:1',
]
HP.video.break_on_level_passed = False

# @launchit.stop

## Initrd

In [37]:
# @launchit.collect_initrd
# @launchit.disable
if CONFIG.exec_mode in [ExecMode.MASTER_NOTEBOOK, ExecMode.LAUNCH_NOTEBOOK] and lu.coalesce(HP.video.capture_env_rams, []):
    assert os.path.exists(CONFIG.initrd_path)
    local_artifact_registry = None
    rams = []
    
    for ram_id in HP.video.capture_env_rams:
        group_id, artifact_id, version = ram_id.split(':')
        
        if local_artifact_registry is None:
            local_artifact_registry = ArtifactRegistry(group_id)
        else:
            assert local_artifact_registry.maven_group_id == group_id
            
        ram_asset = local_artifact_registry.get_asset_content(artifact_id, version, asset_ext='pkl')

        with io.BytesIO(ram_asset) as b:
            ram = pickle.load(b)
            rams.append(ram)

    rams = np.vstack(rams)
    fname = os.path.join(CONFIG.initrd_path, 'capture_env_rams.pkl')

    with open(fname, 'wb') as f:
        pickle.dump(rams, f)
        Logging.get()(f'Capture env RAMs saved to "{fname}"')
#launchit.stop

Env RAMs saved to "/home/misha/dev/mine/neurolab/run/17_rl/initrd-17e_ppo_tr_atari_mp_13/env_rams.pkl"


## Create

In [ ]:
# @launchit.disable_worker
capture_env_rams = None

if lu.coalesce(HP.video.capture_env_rams, []):
    assert os.path.exists(CONFIG.initrd_path)
    fname = os.path.join(CONFIG.initrd_path, 'capture_env_rams.pkl')

    with open(fname, 'rb') as f:
        capture_env_rams = pickle.load(f)

    LOG(f'Capture env RAMs loaded from "{fname}": {len(capture_env_rams)} RAMs')

LS.capture_video_manager = CaptureVideoManager(HP.video.capture_policy, capture_env_rams)

# TRAIN

## RolloutEnvRamSampler

In [28]:
class RolloutEnvRamSampler:
    def __init__(self, rams, patches):
        assert rams is None or isinstance(rams, dict)
        assert patches is None or isinstance(patches, list)
        self.rams = rams
        self.patches = patches

        if self.patches is not None:
            # Verify for all atomic patch there is a patcher func
            for atomic_patch in set(itertools.chain.from_iterable(self.patches)):
                getattr(self, 'patch_' + atomic_patch)

    SampleResult = namedtuple('SampleResult', 'ram, desc')

    def sample(self):
        if self.rams is None or not self.rams:
            return None

        ram_name = RNG.choice(list(self.rams.keys()))
        ram = self.rams[ram_name]
        assert isinstance(ram, np.ndarray)
        patch_desc = ''

        if self.patches is not None and self.patches:
            patch_ind = RNG.choice(len(self.patches))
            patch = self.patches[patch_ind]

            for atomic_patch in patch:
                func = getattr(self, 'patch_' + atomic_patch)
                func(ram)
            
            patch_desc = '+'.join(patch)
        
        return RolloutEnvRamSampler.SampleResult(ram=ram, desc=f'{ram_name}:{patch_desc}')

    def patch_no_score(self, ram):
        ram[73] = 0 
        ram[74] = 0
    
    def patch_last_life(self, ram):
        ram[76] = 0

    def patch_full_igloo(self, ram):
        ram[77] = 15

    def patch_half_igloo(self, ram):
        ram[77] = 7

    def patch_one_remaining_igloo(self, ram):
        ram[77] = 14

    def patch_three_remaining_igloo(self, ram):
        ram[77] = 12

    def patch_temperature_10(self, ram):
        ram[101] = 10

    def patch_temperature_20(self, ram):
        ram[101] = 32

    def patch_bailey_right_at_the_igloo_door(self, ram):
        ram[102] = 124

    def patch_bailey_very_near_igloo_door(self, ram):
        ram[102] = RNG.integers(108, 137, endpoint=True)

    def patch_bailey_near_center(self, ram):
        ram[102] = RNG.integers(45, 95, endpoint=True)

## RolloutGameModifsSampler

In [ ]:
class RolloutGameModifsSampler:
    def __init__(self, game_modifs):
        assert game_modifs is None or isinstance(game_modifs, list)
        self.game_modifs = game_modifs

    SampleResult = namedtuple('SampleResult', 'game_modifs, desc')

    def sample(self):
        if self.game_modifs is None or not self.game_modifs:
            return None

        game_modifs_ind = RNG.choice(len(self.game_modifs))
        game_modifs = self.game_modifs[game_modifs_ind]
        assert isinstance(game_modifs, list)
        return RolloutGameModifsSampler.SampleResult(game_modifs=game_modifs, desc='+'.join(game_modifs))

## Configure

In [ ]:
# @launchit.disable
# @launchit.collect

# Training procedure params (PPO related) 
HP.ppo.global_steps_count = 10_000 # total number of steps 
HP.ppo.rollout_steps_count = 128 # how many steps to run in a single policy rolllout
HP.ppo.rollout_env_rams = [
    'com.develorium.neurolab.frostbite_ram:level1:1',
]
HP.ppo.rollout_env_ram_patches = [
    ['last_life', 'full_igloo', 'bailey_right_at_the_igloo_door', 'temperature_10'],
    ['last_life', 'full_igloo', 'bailey_very_near_igloo_door', 'temperature_10'],  
    ['last_life', 'full_igloo', 'bailey_near_center', 'temperature_10'],
    ['last_life', 'one_remaining_igloo', 'bailey_near_center', 'temperature_10'],
    ['last_life', 'three_remaining_igloo', 'bailey_near_center', 'temperature_20'],
    ['last_life', 'half_igloo', 'bailey_near_center'],
    ['last_life', 'half_igloo'],
    ['last_life'],
]
HP.ppo.rollout_game_modifs = None

HP.ppo.tau = 'const(0.5)' # temperature to inject randomness during actions selection (Gumbel Max)

HP.ppo.epochs_count = 2 
HP.ppo.minibatches_count = 8
HP.ppo.learn_rate = 'const(0.00025)'
HP.ppo.optimizer = 'AdamW'

HP.ppo.vf_coef = 0.5 # coefficient of the value function within loss function
HP.ppo.ent_coef = 'const(0.05)' # coefficient of the entropy member within loss function
HP.ppo.consistency_coef = 0.1 # coefficient of the loss for action plan consistency between adjacent steps
HP.ppo.prediction_coef = 0.1 # coefficient of the loss for observation emb. prediction, if set to 0.0 the prediction loss is not used

HP.ppo.gamma = 0.997 # return discount factor gamma
HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
HP.ppo.clip_vloss = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
HP.ppo.target_kl = None # e target KL divergence threshold
HP.ppo.norm_adv = True # Toggles advantages normalization

# @launchit.stop

In [ ]:
# @launchit.disable_worker
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

## Initrd

In [37]:
# @launchit.collect_initrd
# @launchit.disable
if CONFIG.exec_mode in [ExecMode.MASTER_NOTEBOOK, ExecMode.LAUNCH_NOTEBOOK] and lu.coalesce(HP.ppo.rollout_env_rams, []):
    assert os.path.exists(CONFIG.initrd_path)
    local_artifact_registry = None
    rams = {}
    
    for ram_id in HP.ppo.rollout_env_rams:
        group_id, artifact_id, version = ram_id.split(':')

        if local_artifact_registry is None:
            local_artifact_registry = ArtifactRegistry(group_id)
        else:
            assert local_artifact_registry.maven_group_id == group_id
            
        ram_asset = local_artifact_registry.get_asset_content(artifact_id, version, asset_ext='pkl')

        with io.BytesIO(ram_asset) as b:
            ram = pickle.load(b)
            rams[ram_id] = ram

    fname = os.path.join(CONFIG.initrd_path, 'rollout_env_rams.pkl')

    with open(fname, 'wb') as f:
        pickle.dump(rams, f)
        Logging.get()(f'Rollout env RAMs saved to "{fname}"')
#launchit.stop

Env RAMs saved to "/home/misha/dev/mine/neurolab/run/17_rl/initrd-17e_ppo_tr_atari_mp_13/env_rams.pkl"


## Create

In [ ]:
# @launchit.disable_worker
ump = hp_parse_universal_module(HP.ppo.optimizer)
assert not ump.args
optimizer = getattr(torch.optim, ump.module_name)(LS.agent.parameters(), **ump.kwargs)

ump = hp_parse_universal_module(HP.ppo.learn_rate)
lr_anneal = get_anneal(ump.module_name, *ump.args, **ump.kwargs)

ump = hp_parse_universal_module(HP.ppo.ent_coef)
ent_coef_anneal = get_anneal(ump.module_name, *ump.args, **ump.kwargs)

ump = hp_parse_universal_module(HP.ppo.tau)
tau_anneal = get_anneal(ump.module_name, *ump.args, **ump.kwargs)

In [ ]:
# @launchit.disable_worker
life_stats_afs = dict(l=RecursiveAverageFilter(), r=RecursiveAverageFilter())
episode_stats_afs = dict(l=RecursiveAverageFilter(), r=RecursiveAverageFilter())
clip_fracs_af = RecursiveAverageFilter()

dataset = Dataset(
    envs_count=HP.env.count,
    observation_space_shape=LS.agent.params.observation_space_shape, 
    rollout_steps_count=HP.ppo.rollout_steps_count, 
    obs_sequence_length=LS.agent.params.obs_sequence_length, 
    action_plan_length=LS.agent.params.action_plan_length,
    actions_count=LS.agent.params.actions_count,
    batches_count=HP.ppo.minibatches_count,
    is_shuffled=True,
    is_next_obs=HP.ppo.prediction_coef > 0,
    is_debug=False,
)

for name, v in dataset.shared_tensors.items():
    if v is not None:
        assert v.is_shared(), f'{name} is not shared'
        LOG(f'{name:>28}: {str(v.device):>6}, {str(v.dtype):>15}, {v.shape}')

In [ ]:
# @launchit.disable_worker
rollout_env_rams = None

if lu.coalesce(HP.ppo.rollout_env_rams, []):
    assert os.path.exists(CONFIG.initrd_path)
    fname = os.path.join(CONFIG.initrd_path, 'rollout_env_rams.pkl')

    with open(fname, 'rb') as f:
        rollout_env_rams = pickle.load(f)

    assert len(rollout_env_rams) > 0
    LOG(f'Rollout env RAMs loaded from "{fname}": {len(rollout_env_rams)} RAMs')

rollout_env_ram_sampler = None
rollout_game_modifs_sampler = None
ram = None
game_modifs = None

if rollout_env_rams is not None:
    rollout_env_ram_sampler = RolloutEnvRamSampler(rollout_env_rams, HP.ppo.rollout_env_ram_patches)
    ram = lu.coalesce_fn(rollout_env_ram_sampler.sample(), lambda x: x.ram, None)
elif HP.ppo.rollout_game_modifs is not None:
    rollout_game_modifs_sampler = RolloutGameModifsSampler(HP.ppo.rollout_game_modifs)
    game_modifs = lu.coalesce_fn(rollout_game_modifs_sampler.sample(), lambda x: x.game_modifs, None)

rollout_manager = RolloutManager(
    LS.agent, 
    envs_count=HP.env.count, 
    rollout_steps_count=HP.ppo.rollout_steps_count, 
    prologue_size=dataset.prologue_size,
    random_seed=HP.system.random_seed,
    ram=ram,
    game_modifs=game_modifs,
)

In [ ]:
# @launchit.disable_worker
def report_video_metrics(video_metrics, global_step):
    for metric_name in video_metrics:
        x = video_metrics[metric_name] 
        
        if x:
            metrics_suite[f'game_stats/video/{metric_name}'].extend(x)
            LOG(f'{global_step=}, game_stats/video/{metric_name}={x[-1]}', when=CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK)
            LS.summary_writer.add_scalar(f'game_stats/video/{metric_name}', x[-1], global_step)

## Train

In [ ]:
# @launchit.disable_worker
metrics_suite = defaultdict(list)
pbar = tqdm(total=HP.ppo.global_steps_count)
global_step = 0
global_step_inc = HP.env.count * HP.ppo.rollout_steps_count
is_temporal_action_plan_consistency_loss = LS.agent.params.action_plan_length > 1 and HP.ppo.consistency_coef > 0
is_prediction_loss = HP.ppo.prediction_coef > 0

while global_step < HP.ppo.global_steps_count:
    start_time = time.time()
    global_step_t = global_step / HP.ppo.global_steps_count
    
    # ANNEAL
    lr = lr_anneal(global_step_t)
    for param_group in optimizer.param_groups: param_group["lr"] = lr
    ent_coef = ent_coef_anneal(global_step_t)
    tau = tau_anneal(global_step_t)

    # ROLLOUT
    ram = None
    game_modifs = None

    if rollout_env_ram_sampler is not None:
        ram_sample_result = rollout_env_ram_sampler.sample()
    
        if ram_sample_result is not None:
            ram = ram_sample_result.ram
            LOG(f'RAM for rollout="{ram_sample_result.desc}"')
    elif rollout_game_modifs_sampler is not None:
        game_modifs_sample_result = rollout_game_modifs_sampler.sample()
    
        if game_modifs_sample_result is not None:
            game_modifs = game_modifs_sample_result.game_modifs
            LOG(f'Game modifs for rollout="{game_modifs_sample_result.desc}"')
        
    rr = rollout_manager.rollout(dataset.shared_tensors, tau=tau, ram=ram, game_modifs=game_modifs)

    for stats_item_name, afs in zip(('life_stats', 'episode_stats'), (life_stats_afs, episode_stats_afs)):
        for stats_item in rr.get(stats_item_name, []):
            afs['l'](stats_item['l'])
            afs['r'](stats_item['r'])

    # TRAINING
    for epoch in range(HP.ppo.epochs_count):
        for batch in dataset:
            agent_output = LS.agent(
                obs=batch.obs, 
                padding_masks=batch.obs_pmasks,
                actions=batch.actions, 
                actions_noise=None,
                predict_next_obs_emb=HP.ppo.prediction_coef > 0.0,
                tau=tau,
            )

            assert torch.all(batch.obs_pmasks[:,-1] == 0)
            new_action_log_probs = agent_output.action_log_probs
            new_action_plan_logits = agent_output.action_plan_logits
            new_action_entropies = agent_output.action_entropies
            new_values = agent_output.values
            b_action_log_probs = batch.action_log_probs
            b_values = batch.values
            b_next_action_plan_logits = batch.next_action_plan_logits
            b_dones = batch.dones
            b_advantages = batch.advantages
            b_returns = batch.returns

            # Policy loss
            if HP.ppo.norm_adv:
                b_advantages = (b_advantages - b_advantages.mean()) / (b_advantages.std() + 1e-8)

            logratio = new_action_log_probs - b_action_log_probs
            ratio = torch.exp(logratio)
            pgloss1 = -b_advantages * ratio
            pgloss2 = -b_advantages * torch.clamp(ratio, 1.0 - HP.ppo.clip_coef, 1.0 + HP.ppo.clip_coef)
            pg_loss = torch.max(pgloss1, pgloss2).mean()

            # Value loss
            v_loss_unclipped = (new_values - b_returns) ** 2 
            
            if HP.ppo.clip_vloss:
                v_loss_clipped = b_values + (new_values - b_values).clamp(min=-HP.ppo.clip_coef, max=HP.ppo.clip_coef)
                v_loss = torch.max(v_loss_unclipped, (v_loss_clipped - b_returns) ** 2).mean()
            else:
                v_loss = v_loss_unclipped.mean()

            # Temporal action plan consistency loss
            temporal_action_plan_consistency_loss = 0 
            
            if is_temporal_action_plan_consistency_loss:
                # Take action plans of running only records and compare to next - demand that they should not drift far from each other
                b_running = ~b_dones.bool()
                temporal_action_plan_consistency_loss = F.mse_loss(new_action_plan_logits[b_running,1:], b_next_action_plan_logits[b_running,:-1])

            # Entropy loss
            entropy_loss = new_action_entropies.mean()

            # Next observation prediction loss
            prediction_loss = 0
            
            if is_prediction_loss:
                next_obs_embs = LS.agent.embed_obs(batch.next_obs.unsqueeze(1)).detach().squeeze(1)
                prediction_loss = F.mse_loss(agent_output.next_obs_embs, next_obs_embs)
            
            # Combined losses
            loss = (
                pg_loss 
                + (HP.ppo.vf_coef * v_loss)
                + (HP.ppo.consistency_coef * temporal_action_plan_consistency_loss)
                + (HP.ppo.prediction_coef * prediction_loss)
                - (ent_coef * entropy_loss)
            )

            optimizer.zero_grad()
            loss.backward()
            grad_norm_groups = get_grad_norm_groups(LS.agent, ['cnn', 'transformer', 'actor', 'critic', 'predictor'])
            grad_norm = torch.nn.utils.clip_grad_norm_(LS.agent.parameters(), max_norm=HP.ppo.max_grad_norm)
            optimizer.step()

            with torch.no_grad():
                # calculate approx_kl http://joschu.net/blog/kl-approx.html
                approx_kl = ((ratio - 1) - logratio).mean()
                clip_fracs_af(((ratio - 1.0).abs() > HP.ppo.clip_coef).float().mean().item())

        if HP.ppo.target_kl is not None and approx_kl > HP.ppo.target_kl:
            break

    var_y = torch.var(dataset.returns)
    explained_var = lu.when(var_y == 0, np.nan, 1 - torch.var(dataset.returns - dataset.values) / var_y)

    # REPORT
    LS.summary_writer.add_scalar('charts/sps', global_step_inc / (time.time() - start_time), global_step)
    
    LS.summary_writer.add_scalar('curriculum/learning_rate', lr, global_step)
    LS.summary_writer.add_scalar('curriculum/entropy_coefficient', ent_coef, global_step)
    LS.summary_writer.add_scalar('curriculum/tau', tau, global_step)
    
    LS.summary_writer.add_scalar('grad/grad_norm', grad_norm, global_step)

    for grad_norm_group_name, grad_norm_group_value in grad_norm_groups.items():
        LS.summary_writer.add_scalar(f'grad/grad_norm_{grad_norm_group_name}', grad_norm_group_value, global_step)
    
    LS.summary_writer.add_scalar('losses/loss', loss, global_step)
    LS.summary_writer.add_scalar('losses/policy_loss', pg_loss, global_step)
    LS.summary_writer.add_scalar('losses/value_loss', v_loss, global_step)
    LS.summary_writer.add_scalar('losses/entropy_loss', entropy_loss, global_step)

    if is_temporal_action_plan_consistency_loss:
        LS.summary_writer.add_scalar('losses/temporal_action_plan_consistency_loss', temporal_action_plan_consistency_loss, global_step)

    if is_prediction_loss:
        LS.summary_writer.add_scalar('losses/prediction_loss', prediction_loss, global_step)
        
    LS.summary_writer.add_scalar('losses/approx_kl', approx_kl, global_step)
    LS.summary_writer.add_scalar('losses/clipfrac', clip_fracs_af.reset(), global_step)
    LS.summary_writer.add_scalar('losses/explained_variance', explained_var, global_step)

    if episode_stats_afs['r'].n > 0:
        assert episode_stats_afs['l'].n == episode_stats_afs['r'].n
        l = episode_stats_afs['l'].reset()
        r = episode_stats_afs['r'].reset()
        LS.summary_writer.add_scalar('game_stats/envs/episode_l', l, global_step)
        LS.summary_writer.add_scalar('game_stats/envs/episode_r', r, global_step)
        metrics_suite['game_stats/envs/episode_l'].append(l)
        metrics_suite['game_stats/envs/episode_r'].append(r)

    if life_stats_afs['r'].n > 0:
        assert life_stats_afs['l'].n == life_stats_afs['r'].n
        l = life_stats_afs['l'].reset()
        r = life_stats_afs['r'].reset()
        LS.summary_writer.add_scalar('game_stats/envs/life_l', l, global_step)
        LS.summary_writer.add_scalar('game_stats/envs/life_r', r, global_step)
        metrics_suite['game_stats/envs/life_l'].append(l)
        metrics_suite['game_stats/envs/life_r'].append(r)

    LS.summary_writer.add_scalar('rollout/value_mean', dataset.values.mean(), global_step)
    LS.summary_writer.add_scalar('rollout/advantage_mean', dataset.advantages.mean(), global_step)

    # VIDEO
    if LS.capture_video_manager.should_capture_video(global_step, is_last_step=global_step + global_step_inc >= HP.ppo.global_steps_count):
        if LS.capture_video_manager.is_busy():
            video_metrics = LS.capture_video_manager.upload_captured_video(is_drain=True)
            report_video_metrics(video_metrics, global_step)

        LS.capture_video_manager.schedule_capture_video(tau=tau, global_step=global_step)

    video_metrics = LS.capture_video_manager.upload_captured_video()
    report_video_metrics(video_metrics, global_step)
    LS.summary_writer.flush()

    # min is used to not overflow progress bar at the end (global_step could get > HP.ppo.global_step_size)
    pbar.update(min(global_step_inc, HP.ppo.global_steps_count - global_step)) 
    global_step += global_step_inc
    LOG(f'Progress {100 * global_step / HP.ppo.global_steps_count:.2f}% ({global_step:_} / {HP.ppo.global_steps_count:_})', 
        when=CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK)

video_metrics = LS.capture_video_manager.upload_captured_video(is_drain=True)
report_video_metrics(video_metrics, global_step)
LS.summary_writer.flush()
pbar.close()

## Save

In [ ]:
# @launchit.disable_worker
lc = HP.launch_component()

if lc.version != 0:
    with io.BytesIO() as b:
        torch.save(LS.agent.state_dict(), b)
        LS.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='pt', asset_classifier='agent', replace=True)
    
    with io.StringIO() as b:
        json.dump(dataclasses.asdict(LS.agent.params), b)
        LS.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='agent_params', replace=True)

    with io.StringIO() as b:
        json.dump(metrics_suite, b)
        LS.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='metrics_suite', replace=True)

    with open(CONFIG.metrics_suite_fname, 'w') as f:
        json.dump(metrics_suite, f)
        LOG(f'Metrics suite saved to "{CONFIG.metrics_suite_fname}"')

# LaunchIt!

## LAUNCH_NOTEBOOK

In [ ]:
# @launchit.disable
launchit_t0 = time.time()

In [ ]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    lc = HP.launch_component()
    component_version = int(Autoincrement.get(lc.uri))
    assert component_version > 0, component_version
    LS.artifact_registry.register_component(lc.name, component_version)
    LOG(f'Model instance registered, version={component_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=component_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN.value,
    )
    launch_notebook_fname = launchit.launchit(
        CONFIG.self_fname, 
        launch_serial=component_version, 
        expandvars=expandvars, 
        collect_inds=['initrd', 'temp_config'], 
        disable_inds=[]
    )
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

## DOCKER_LAUNCH_NOTEBOOK

In [30]:
# @launchit.disable
launchit_t0 = time.time()

In [31]:
# @launchit.collect_manual_run_docker_launch
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK:
    import launch_dispatcher
    image_tag = os.path.join(CONFIG.docker_registry, '${MODEL_NAME}' + ':' + '${MODEL_VERSION}')
    launch_request = dict(
        launch_image=image_tag,
        keep_container=False,
    )
    launch_dispatcher.LaunchRequest.run(launch_request)
# @launchit.stop

In [32]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    lc = HP.launch_component()
    component_version = int(Autoincrement.get(lc.uri))
    assert component_version > 0, component_version
    LS.artifact_registry.register_component(lc.name, component_version)
    LOG(f'Model instance registered, version={component_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=component_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN.value,
    )
    launch_notebook_fname = launchit.launchit(
        CONFIG.self_fname, 
        launch_serial=component_version, 
        expandvars=expandvars, 
        collect_inds=['initrd', 'build_docker_launch', 'manual_run_docker_launch', 'temp_config'], 
        disable_inds=[]
    )
    LOG(f'Created docker launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=3
Creating /home/misha/dev/mine/neurolab/17_rl/17e_ppo_tr_atari_mp_13-launch3.ipynb
Created docker launch notebook "/home/misha/dev/mine/neurolab/17_rl/17e_ppo_tr_atari_mp_13-launch3.ipynb"


## Optuna (model selection)

### Templates

In [15]:
# @launchit.collect_optuna
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK and os.path.exists('${OPTUNA_STUDY_FNAME}'):
    import re
    import launchit
    optuna_study_fname = '${OPTUNA_STUDY_FNAME}'
    optuna_study_storage = JournalStorage(JournalFileBackend(optuna_study_fname))
    optuna_study_name = '${OPTUNA_STUDY_NAME}'
    assert optuna_study_name
    assert optuna_study_name != '$' + '{OPTUNA_STUDY_NAME}', optuna_study_name
    optuna_study = optuna.load_study(study_name=optuna_study_name, storage=optuna_study_storage)
    optuna_trial = optuna_study.ask()
    optuna_study_serial = optuna_study.user_attrs['STUDY_SERIAL']
    optuna_study_nb_fname = re.sub('.optuna$', '.ipynb', optuna_study_fname)

    source_code = launchit.extract_source_code(optuna_study_nb_fname)
    Logging.get()(f'Extracted source code for set_hyperparamters from "{optuna_study_nb_fname}"')
    module = lu.make_module('optuna_study_hyperparameters', source_code)
    HP = module.set_hyperparameters(Hyperparameters(), optuna_study, optuna_trial)
    Logging.get()(f'HP=\n{pprint.pformat(HP._asdict(), sort_dicts=False)}\n')

    assert os.path.exists(CONFIG.initrd_path)

    with open(os.path.join(CONFIG.initrd_path, 'optuna_trial.json'), 'w') as f:
        optuna_trial_dict = dict(
            trial_number=optuna_trial.number,
            study_serial=optuna_study_serial,
            study_name=optuna_study_name,
        )
        json.dump(optuna_trial_dict, f)

    with open(os.path.join(CONFIG.initrd_path, 'hyperparameters.json'), 'w') as f:
        json.dump(HP._asdict(), f)
        
elif CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK:
    hyperparameters_fname = os.path.join(CONFIG.initrd_path, 'hyperparameters.json')
    
    with open(hyperparameters_fname, 'r') as f:
        HP = Hyperparameters.from_dict(json.load(f))
        Logging.get()(f'HP loaded from "{hyperparameters_fname}"')
        Logging.get()(f'HP=\n{pprint.pformat(HP._asdict(), sort_dicts=False)}\n')


### optuna_run_docker_launch

In [16]:
# @launchit.collect_optuna_run_docker_launch
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK:
    # optuna_study and optuna_trial are created in cell above
    assert optuna_study is not None
    assert optuna_trial is not None

    import launch_dispatcher
    short_image_tag = '${MODEL_NAME}' + ':' + '${MODEL_VERSION}'
    image_tag = os.path.join(CONFIG.docker_registry, short_image_tag)
    launch_request = dict(
        launch_image=image_tag,
        result_fname=os.path.join(project_root_path, CONFIG.relative_metrics_suite_fname),
        keep_container=False,
    )
    launch_result_metadata, launch_result_body = launch_dispatcher.LaunchRequest.run(launch_request)
    
    with open(CONFIG.self_fname + '.out', mode='wt') as out_file:
        if launch_result_metadata['is_ok']:
            if launch_result_body:
                decisive_metric = 'game_stats/video/reward'
                
                with io.BytesIO(launch_result_body) as b:
                    launch_result_dict = json.load(b)
    
                if launch_result_dict.get(decisive_metric, []):
                    scalar_result = np.array(launch_result_dict[decisive_metric]).mean() # reduce to just scalar e.g. via mean, sum, last
                    optuna_study.tell(optuna_trial, scalar_result, state=optuna.trial.TrialState.COMPLETE)
    
                    message = f'Trial {optuna_trial.number} ({short_image_tag}) finished with value: {scalar_result} and parameters: {optuna_trial.params}'
                    
                    try:
                        best_trial = optuna_study._get_best_trial(deepcopy=False)
                        message += f'. Best is trial {best_trial.number} with value: {best_trial.value}'
                    except ValueError:
                        # If no feasible trials are completed yet, study.best_trial raises ValueError
                        pass

                    out_file.write(message)
                else:
                    optuna_study.tell(optuna_trial, state=optuna.trial.TrialState.FAIL)
                    out_file.write(f'Trial {optuna_trial.number} ({short_image_tag}) finished with empty/missing "{decisive_metric}"')
            else:
                optuna_study.tell(optuna_trial, state=optuna.trial.TrialState.FAIL)
                out_file.write(f'Trial {optuna_trial.number} ({short_image_tag}) finished with empty result body: {launch_result_metadata=}')
        else:
            optuna_study.tell(optuna_trial, state=optuna.trial.TrialState.FAIL)
            out_file.write(f'Trial {optuna_trial.number} ({short_image_tag}) failed: {launch_result_metadata=}')

    import warnings
    warnings.filterwarnings('ignore', category=UserWarning, message="To exit: use 'exit'")
    sys.exit(0)

### Unleash

In [ ]:
# @launchit.disable
import concurrent.futures as cf
import subprocess

# Executed in a separate thread with GIL locked
def run_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.self_name}'))
    assert model_version > 0, model_version
    LS.artifact_registry.register_component(CONFIG.self_name, model_version)
    LOG(f'Model instance registered, version={model_version}')

    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN.value,
        OPTUNA_STUDY_FNAME=optuna_study_fname,
        OPTUNA_STUDY_NAME=optuna_study_name,
    )
    launch_fname = launchit.launchit(
        CONFIG.self_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of docker a launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())

optuna_study_name = '17e_study_16.1'
optuna_study_serial = re.match(r'\w+_([\d\.]+)', optuna_study_name).group(1)
optuna_study_fname = os.path.join(CONFIG.subproject_path, 'optuna', optuna_study_name, optuna_study_name + '.optuna')
grid_search_space = None
optuna_study = optuna.create_study(
    study_name=optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True,
    sampler=lu.when(grid_search_space, lambda: optuna.samplers.GridSampler(grid_search_space), None),
)
optuna_study.set_user_attr('STUDY_SERIAL', optuna_study_serial)
launches_count = 3 * 8

with cf.ThreadPoolExecutor(max_workers=min(9, launches_count)) as executor:
    futures = {executor.submit(run_optuna_launch): i for i in range(launches_count)}

    # Consume done launches and launch pending ones
    for future in cf.as_completed(futures):
        launch_index = futures[future]
        future.result()

[I 2026-07-05 21:06:49,141] A new study created in Journal with name: 17e_study_16.1


Model instance registered, version=238
Model instance registered, version=236
Creating /home/misha/dev/mine/neurolab/run/17_rl/17e_ppo_tr_atari_mp_13-launch238.ipynb
Model instance registered, version=237
Model instance registered, version=239
Model instance registered, version=241
Model instance registered, version=244
Model instance registered, version=240
Model instance registered, version=242
Creating /home/misha/dev/mine/neurolab/run/17_rl/17e_ppo_tr_atari_mp_13-launch236.ipynb
Model instance registered, version=243
Creating /home/misha/dev/mine/neurolab/run/17_rl/17e_ppo_tr_atari_mp_13-launch237.ipynb
Creating /home/misha/dev/mine/neurolab/run/17_rl/17e_ppo_tr_atari_mp_13-launch239.ipynb
Creating /home/misha/dev/mine/neurolab/run/17_rl/17e_ppo_tr_atari_mp_13-launch241.ipynb
Creating /home/misha/dev/mine/neurolab/run/17_rl/17e_ppo_tr_atari_mp_13-launch244.ipynb
Creating /home/misha/dev/mine/neurolab/run/17_rl/17e_ppo_tr_atari_mp_13-launch240.ipynb
Creating /home/misha/dev/mine/neu

In [ ]:
# @launchit.disable
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs['MODEL_VERSION']}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

# Docker build

In [71]:
# @launchit.collect_build_docker_launch
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK:
    import subprocess
    import tempfile

    assert os.path.exists(os.path.join(build_project_root_path, CONFIG.relative_self_fname))

    dockerfile_content = f'''
FROM {os.path.join(CONFIG.docker_registry, 'neurolab_source:latest')}
USER 0
RUN pip install --break-system-packages --no-cache-dir ale_py==0.12.0+neurolab --index-url=http://nexus:8081/repository/neurolab-pypi/simple --trusted-host=nexus
USER 1000
WORKDIR /neurolab
RUN git pull --rebase
WORKDIR /neurolab/{CONFIG.subproject_name}
COPY --chown=1000:1000 {CONFIG.relative_self_fname} .
'''
    if os.path.exists(CONFIG.initrd_path):
        dockerfile_content += f'''
RUN mkdir -p /neurolab/{CONFIG.relative_initrd_path}
COPY --chown=1000:1000  {CONFIG.relative_initrd_path} /neurolab/{CONFIG.relative_initrd_path}
'''

    dockerfile_content += f'''
RUN touch /neurolab/.docker_launch
CMD ["papermill", "{os.path.basename(CONFIG.self_fname)}", "{os.path.basename(CONFIG.self_fname)}"]
    '''
    with tempfile.NamedTemporaryFile(mode='wt', suffix='.Dockerfile') as dockerfile:
        with open(dockerfile.name, 'w') as f:
            f.write(dockerfile_content)

        image_tag = os.path.join(CONFIG.docker_registry, '${MODEL_NAME}' + ':' + '${MODEL_VERSION}')
        subprocess.run(
            ['docker', 'buildx', 'build', '-f', dockerfile.name, '-t', image_tag, build_project_root_path, '--load', '--no-cache', '--network=host'],
            text=True,                # Handles input/output as strings instead of bytes
            capture_output=False,     # Streams Docker's build output directly to your terminal
            check=True                # Raises an exception if the command fails
        )
        subprocess.run(
            ['docker', 'push', image_tag],
            text=True,                # Handles input/output as strings instead of bytes
            capture_output=False,     # Streams Docker's build output directly to your terminal
            check=True                # Raises an exception if the command fails
        )
# @launchit.stop